# Feature Selection and Optimization - Hybrid Multi-Stage Approach

**Project:** Solar Data Science Capstone - Ketura Solar Plant  
**Author:** Data Science Team  
**Date:** December 2025  
**Objective:** Select optimal feature subset for Clear Sky Index (target_k) prediction using multi-day ahead forecasting (D+1 to D+5)

---

## Scientific Approach: Hybrid Multi-Stage Selection

This notebook implements a **rigorous, academically sound** feature selection pipeline that combines:

### 1️⃣ **Filter Methods** (Statistical Screening)
- **Variance Thresholding**: Remove uninformative features (variance < 0.001)
- **Mutual Information (MI)**: Measure non-linear correlation with target
- Removes ~30% of features while retaining information

### 2️⃣ **Domain Knowledge Constraints** (Physics-Driven)
- **Solar Geometry**: Ensure critical astronomical features (solar position, POA irradiance)
- **Weather Forecasts**: Guarantee all NWP inputs (GFS forecasts)
- **Temporal Dependencies**: Protect lag and rolling features
- Prevents elimination of physically essential features

### 3️⃣ **Wrapper Methods** (Model-Based Selection)
- **Recursive Feature Elimination (RFE)** with XGBoost
- **Time-Series Cross-Validation** (5-fold, temporally ordered)
- Iterative removal of weakest predictors

### 4️⃣ **SHAP Validation** (Explainability)
- **Shapley Additive Explanations** for feature importance
- Validates that selected features have meaningful impact
- Provides interpretability for stakeholders

---

## Academic Justification

This approach addresses key challenges in solar forecasting feature selection:

1. **Statistical Significance**: MI filtering ensures non-linear correlation (Vergara & Estévez, 2014)
2. **Physical Relevance**: Domain constraints prevent elimination of known causal factors (Inman et al., 2013)
3. **Temporal Integrity**: Time-series CV prevents data leakage (Bergmeir & Benítez, 2012)
4. **Model Interpretability**: SHAP provides local and global explanations (Lundberg & Lee, 2017)

### Key References:
- Inman, R. H., Pedro, H. T., & Coimbra, C. F. (2013). *Solar forecasting methods for renewable energy integration*. Progress in Energy and Combustion Science.
- Lundberg, S. M., & Lee, S. I. (2017). *A unified approach to interpreting model predictions*. NeurIPS.
- Vergara, J. R., & Estévez, P. A. (2014). *A review of feature selection methods based on mutual information*. Neural Computing and Applications.

---

## Notebook Structure

**Setup & Configuration:**
- [Cell 0](#cell-0) - Imports & Environment
- [Cell 1](#cell-1) - Configuration & Data Loading

**Stage 1: Filter Methods**
- [Cell 2](#cell-2) - Variance Thresholding & Mutual Information

**Stage 2: Domain Knowledge**
- [Cell 3](#cell-3) - Core Feature Protection

**Stage 3: Wrapper Methods**
- [Cell 4](#cell-4) - Recursive Feature Elimination (RFE)

**Stage 4: Validation**
- [Cell 5](#cell-5) - SHAP Analysis & Interpretability
- [Cell 6](#cell-6) - Final Validation & Export

---

## Expected Outputs
```
artifacts/05_feature_selection/
├── final_selected_features.json          # 48-52 features
├── feature_importance_shap.csv           # SHAP values
└── selection_pipeline_config.json        # Reproducibility

outputs/05_feature_selection/
├── reports/
│   ├── stage1_mi_scores.csv
│   ├── stage3_rfe_cv_results.csv
│   ├── stage4_shap_importance.csv
│   └── feature_selection_report.md
└── plots/
    ├── stage1_mutual_information.png
    ├── stage3_rfe_convergence.png
    ├── stage4_shap_summary.png
    └── final_feature_distribution.png
```

<a id='cell-0'></a>
## Cell 0: Imports & Environment Setup

Load all required libraries and configure the computational environment.

In [1]:
# ============================================================================
# Cell 0: Imports & Environment Setup
# ============================================================================

# === Standard Library ===
import os
import sys
import json
import warnings
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple

# === Data Handling ===
import numpy as np
import pandas as pd
from scipy import stats

# === Machine Learning ===
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    RFE, 
    mutual_info_regression,
    VarianceThreshold
)
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# === Interpretability ===
import shap

# === Visualization ===
import matplotlib.pyplot as plt
import seaborn as sns

# === Configuration ===
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.precision', 4)

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Environment configured")
print(f"  Python: {sys.version.split()[0]}")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")
print(f"  scikit-learn: {__import__('sklearn').__version__}")
print(f"  xgboost: {xgb.__version__}")
print(f"  shap: {shap.__version__}")
print(f"  Random seed: {RANDOM_SEED}")

✓ Environment configured
  Python: 3.11.5
  pandas: 2.0.3
  numpy: 1.24.3
  scikit-learn: 1.3.0
  xgboost: 3.1.2
  shap: 0.48.0
  Random seed: 42


<a id='cell-1'></a>
## Cell 1: Configuration & Data Loading

### Configuration Strategy

The configuration defines:
1. **Core Features** (domain knowledge) - features that MUST be retained
2. **Category Minimums** - balanced representation across feature types
3. **Statistical Thresholds** - MI, variance, RFE parameters

### Core Feature Categories:

**Solar Geometry** (astronomical, deterministic):
- Solar position angles (zenith, azimuth, elevation)
- Plane-of-Array (POA) irradiance components
- Angle of Incidence (AOI)
- Clear-sky models (baseline expectation)

**Weather Forecasts** (NWP inputs, critical for real-time prediction):
- GFS forecast components: GHI, DNI, DHI
- Temperature, wind speed, cloud cover
- These are the ONLY external inputs available at forecast time

**Temporal Dependencies** (historical patterns):
- Lag features: 1d, 2d, 3d, 7d (multi-horizon)
- Rolling statistics: same-time-of-day patterns
- Essential for time-series forecasting

**Performance Indicators** (system state):
- Efficiency metrics
- Performance ratios
- Operational status indicators

In [2]:
# ============================================================================
# Cell 1: Configuration & Data Loading (FIXED - Preserve Lag Features)
# ============================================================================

print("="*80)
print("FEATURE SELECTION - HYBRID MULTI-STAGE APPROACH")
print("="*80)

# ============================================================================
# 1. Automatic File Discovery
# ============================================================================

TARGET_FILENAME = 'features_1H_target_k.parquet'

print(f"\n🔍 Searching for '{TARGET_FILENAME}'...")

# Find project root
current_path = Path.cwd()
project_root = None

for parent in [current_path] + list(current_path.parents):
    if parent.name == 'solar_ds_capstone':
        project_root = parent
        break

if project_root is None:
    project_root = current_path.parent

print(f"📂 Project Root: {project_root}")

# Recursive search for data file
found_path = None
for root, dirs, files in os.walk(project_root):
    if TARGET_FILENAME in files:
        found_path = Path(root) / TARGET_FILENAME
        break

if found_path is None:
    raise FileNotFoundError(f"❌ Could not find '{TARGET_FILENAME}' in project!")

print(f"✅ Found data file: {found_path}")

# ============================================================================
# 2. Enhanced Configuration with Core Features
# ============================================================================

CONFIG = {
    # === Paths ===
    'data_path': found_path,
    'artifacts_dir': project_root / 'artifacts' / '05_feature_selection',
    'reports_dir': project_root / 'outputs' / '05_feature_selection' / 'reports',
    'plots_dir': project_root / 'outputs' / '05_feature_selection' / 'plots',
    
    # === Target ===
    'target_column': 'target_k',
    
    # === Core Features (MUST KEEP - Domain Knowledge) ===
    'core_features': {
        'solar_geometry': [
            'solar_zenith',
            'solar_azimuth', 
            'sun_elevation',
            'aoi',
            'time_to_solar_noon'
        ],
        'poa_clearsky': [
            'poa_global_clearsky',
            'poa_direct_clearsky',
            'poa_diffuse_clearsky'
        ],
        'forecasts_gfs': [
            'fcst_ghi',
            'fcst_dni',
            'fcst_dhi',
            'fcst_temp_2m',
            'fcst_wind_speed',
            'fcst_cloud_cover'
        ],
        'clearsky_baseline': [
            'ghi_clearsky',
            'dni_clearsky',
            'dhi_clearsky'
        ]
    },
    
    # === Statistical Thresholds ===
    'variance_threshold': 0.001,
    'mi_threshold': 0.01,
    'rfe_n_features': 50,  # Target (increased from 40)
    'rfe_step': 0.05,
    
    # === Minimum Features Per Category (UPDATED) ===
    'min_features_per_category': {
        'lag': 12,           # ↑ from 8 (need multi-horizon)
        'rolling': 8,        # ↑ from 5 (mean + std important)
        'solar': 10,         # ↑ from 5 (includes POA!)
        'weather': 8,        # ↑ from 5 (all forecasts)
        'temporal': 5,       # ↑ from 3 (cyclic + solar time)
        'performance': 2,    # NEW category
        'other': 5           # Miscellaneous important features
    },
    
    # === Model Parameters ===
    'xgb_params': {
        'n_estimators': 100,
        'max_depth': 6,       # Slightly increased for complexity
        'learning_rate': 0.1,
        'subsample': 0.8,     # Prevent overfitting
        'colsample_bytree': 0.8,
        'random_state': 42,
        'n_jobs': -1,
        'objective': 'reg:squarederror'
    },
    
    # === Cross-Validation ===
    'cv_splits': 5,
    'test_size': 0.2,
    'random_seed': 42,
    
    # === SHAP Configuration ===
    'shap_sample_size': 1000,  # For computational efficiency
    'shap_threshold': 0.001    # Minimum mean |SHAP| value
}

# Create output directories
for dir_key in ['artifacts_dir', 'reports_dir', 'plots_dir']:
    CONFIG[dir_key].mkdir(parents=True, exist_ok=True)

print(f"\n📁 Output directories created:")
print(f"   Artifacts: {CONFIG['artifacts_dir']}")
print(f"   Reports: {CONFIG['reports_dir']}")
print(f"   Plots: {CONFIG['plots_dir']}")

# Flatten core features for easy access
CORE_FEATURES_FLAT = []
for category, features in CONFIG['core_features'].items():
    CORE_FEATURES_FLAT.extend(features)

print(f"\n🔒 Core features (protected): {len(CORE_FEATURES_FLAT)}")
for cat, feats in CONFIG['core_features'].items():
    print(f"   {cat}: {len(feats)} features")

# ============================================================================
# 3. Load Data
# ============================================================================

print(f"\n📊 Loading data from: {CONFIG['data_path'].name}")

df = pd.read_parquet(CONFIG['data_path'])

print(f"   Shape: {df.shape}")
print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"   Date range: {df.index.min()} to {df.index.max()}")

# ============================================================================
# 4. Define Target and Features
# ============================================================================

TARGET_COL = CONFIG['target_column']

if TARGET_COL not in df.columns:
    raise ValueError(f"❌ Target '{TARGET_COL}' not found!")

print(f"\n🎯 Target: {TARGET_COL}")
print(f"   Mean: {df[TARGET_COL].mean():.4f}")
print(f"   Std: {df[TARGET_COL].std():.4f}")
print(f"   Range: [{df[TARGET_COL].min():.4f}, {df[TARGET_COL].max():.4f}]")
print(f"   Missing: {df[TARGET_COL].isna().sum()} ({df[TARGET_COL].isna().mean()*100:.2f}%)")

# ============================================================================
# 5. Identify Feature Columns (FIXED: Preserve Lag & Rolling Features)
# ============================================================================

print(f"\n🔍 Identifying feature columns...")

# Columns to EXCLUDE (targets, metadata, leakage risk)
EXCLUDE_COLS = [
    TARGET_COL,
    'timestamp',
    'theoretical_power',  # ⚠️ CRITICAL: Do NOT use as input!
    'site_total_AC_kW',
    'site_total_DC_kW',
    's01_inv1_AC_kW',     # Direct AC power (current)
    's01_inv2_AC_kW',     # Direct AC power (current)
    's01_inv1_DC_kW',     # Direct DC power (current)
    's01_inv2_DC_kW'      # Direct DC power (current)
]

feature_columns = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f"   Total columns: {len(df.columns)}")
print(f"   Excluded: {len(EXCLUDE_COLS)}")
print(f"   Initial features: {len(feature_columns)}")

# ============================================================================
# CRITICAL FIX: Smart Leakage Detection
# Only remove DIRECT power measurements, NOT lag/rolling features
# ============================================================================

print(f"\n   🔍 Checking for leakage features...")

# Identify power columns that are NOT lag or rolling features
leakage_candidates = [
    c for c in feature_columns 
    if any(x in c for x in ['_AC_kW', '_DC_kW', 'site_total'])
]

# Separate into safe (historical) and unsafe (current) features
safe_historical = [c for c in leakage_candidates 
                   if 'lag' in c or 'rolling' in c]

unsafe_current = [c for c in leakage_candidates 
                  if c not in safe_historical]

if unsafe_current:
    print(f"   ⚠️ Found {len(unsafe_current)} current power features (leakage risk):")
    for pc in unsafe_current[:5]:
        print(f"      ❌ {pc}")
    
    # Remove only the unsafe ones
    feature_columns = [c for c in feature_columns if c not in unsafe_current]
    print(f"   ✅ Removed {len(unsafe_current)} leakage features")

if safe_historical:
    print(f"\n   ✅ Preserved {len(safe_historical)} historical features (lag/rolling):")
    for sf in safe_historical[:5]:
        print(f"      ✓ {sf}")
    if len(safe_historical) > 5:
        print(f"      ... and {len(safe_historical)-5} more")

print(f"\n   Final feature count: {len(feature_columns)}")

# ============================================================================
# 6. Prepare X and y
# ============================================================================

print(f"\n🔧 Preparing feature matrix...")

X = df[feature_columns].copy()
y = df[TARGET_COL].copy()

# Remove rows with missing target
valid_mask = y.notna()
X = X[valid_mask]
y = y[valid_mask]

print(f"   Valid rows: {len(X):,} / {len(df):,} ({len(X)/len(df)*100:.1f}%)")

# Handle missing values in features
missing_counts = X.isna().sum()
features_with_missing = missing_counts[missing_counts > 0]

if len(features_with_missing) > 0:
    print(f"\n   ⚠️ Found {len(features_with_missing)} features with missing values")
    print(f"      Total missing: {missing_counts.sum():,}")
    
    # Time-series safe imputation (forward fill, then backward fill)
    X = X.fillna(method='ffill').fillna(method='bfill')
    
    remaining = X.isna().sum().sum()
    if remaining > 0:
        print(f"      Still {remaining} missing after ffill/bfill")
        X = X.fillna(X.median())
    
    print(f"      ✅ All missing values handled")

# ============================================================================
# 7. Feature Categorization Function
# ============================================================================

def categorize_feature(name: str) -> str:
    """
    Categorize features for balanced selection.
    
    Categories:
    - lag: Historical lag features (1d, 2d, 7d, etc.)
    - rolling: Same-time-of-day rolling statistics
    - solar: Solar geometry and position (includes POA!)
    - weather: Weather forecasts and measurements
    - temporal: Time-based features (hour, day, month)
    - performance: System performance indicators
    - other: Miscellaneous features
    """
    name_lower = name.lower()
    
    # Lag features (but not rolling)
    if 'lag' in name_lower and 'rolling' not in name_lower:
        return 'lag'
    
    # Rolling statistics
    elif 'rolling' in name_lower:
        return 'rolling'
    
    # Solar geometry (INCLUDES POA!)
    elif any(x in name_lower for x in [
        'sun', 'solar', 'azimuth', 'elevation', 'zenith',
        'airmass', 'aoi', 'poa_', 'clearsky'
    ]):
        return 'solar'
    
    # Weather (forecasts + measurements)
    elif any(x in name_lower for x in [
        'fcst_', 'temp', 'wind', 'ghi', 'dni', 'dhi', 'cloud'
    ]):
        return 'weather'
    
    # Temporal
    elif any(x in name_lower for x in [
        'hour', 'month', 'day', 'doy', '_sin', '_cos', 'season'
    ]):
        return 'temporal'
    
    # Performance
    elif any(x in name_lower for x in [
        'efficiency', 'performance', 'ratio', 'loading'
    ]):
        return 'performance'
    
    # Other
    else:
        return 'other'

# Categorize all features
feature_categories = {f: categorize_feature(f) for f in X.columns}
category_counts = Counter(feature_categories.values())

print(f"\n📊 Initial Feature Distribution:")
for cat in sorted(category_counts.keys()):
    count = category_counts[cat]
    min_req = CONFIG['min_features_per_category'].get(cat, 0)
    status = "✅" if count >= min_req else "⚠️"
    print(f"   {status} {cat}: {count} (min required: {min_req})")

# ============================================================================
# 8. Validate Core Features Exist
# ============================================================================

print(f"\n🔒 Validating core features...")

missing_core = []
for cat, feats in CONFIG['core_features'].items():
    for feat in feats:
        if feat not in X.columns:
            missing_core.append((cat, feat))

if missing_core:
    print(f"   ⚠️ WARNING: {len(missing_core)} core features not found:")
    for cat, feat in missing_core[:5]:
        print(f"      - {feat} ({cat})")
    if len(missing_core) > 5:
        print(f"      ... and {len(missing_core)-5} more")
    print(f"   These will be skipped in protection logic.")
else:
    print(f"   ✅ All {len(CORE_FEATURES_FLAT)} core features present")

# Update CORE_FEATURES_FLAT to only include existing features
CORE_FEATURES_FLAT = [f for f in CORE_FEATURES_FLAT if f in X.columns]
print(f"   Protected features: {len(CORE_FEATURES_FLAT)}")

# ============================================================================
# 9. Save Configuration
# ============================================================================

config_save = {k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}
config_save['core_features_count'] = len(CORE_FEATURES_FLAT)
config_save['initial_feature_count'] = len(X.columns)
config_save['initial_sample_count'] = len(X)
config_save['category_distribution'] = dict(category_counts)

with open(CONFIG['artifacts_dir'] / 'config_runtime.json', 'w') as f:
    json.dump(config_save, f, indent=2)

print(f"\n💾 Configuration saved: config_runtime.json")

# ============================================================================
# 10. Summary
# ============================================================================

print(f"\n{'='*80}")
print("CONFIGURATION COMPLETE")
print(f"{'='*80}")
print(f"✅ Data loaded: {X.shape[0]:,} samples × {X.shape[1]} features")
print(f"✅ Target: {TARGET_COL}")
print(f"✅ Core features protected: {len(CORE_FEATURES_FLAT)}")
print(f"✅ Lag features preserved: {category_counts['lag']}")
print(f"✅ Rolling features preserved: {category_counts['rolling']}")
print(f"✅ Ready for Stage 1 (Filter Methods)")
print(f"{'='*80}\n")

FEATURE SELECTION - HYBRID MULTI-STAGE APPROACH

🔍 Searching for 'features_1H_target_k.parquet'...
📂 Project Root: C:\Users\baraky\PycharmProjects\solar_ds_capstone
✅ Found data file: C:\Users\baraky\PycharmProjects\solar_ds_capstone\04_feature_engineering\data\processed\04_features_target_k\features_1H_target_k.parquet

📁 Output directories created:
   Artifacts: C:\Users\baraky\PycharmProjects\solar_ds_capstone\artifacts\05_feature_selection
   Reports: C:\Users\baraky\PycharmProjects\solar_ds_capstone\outputs\05_feature_selection\reports
   Plots: C:\Users\baraky\PycharmProjects\solar_ds_capstone\outputs\05_feature_selection\plots

🔒 Core features (protected): 17
   solar_geometry: 5 features
   poa_clearsky: 3 features
   forecasts_gfs: 6 features
   clearsky_baseline: 3 features

📊 Loading data from: features_1H_target_k.parquet
   Shape: (32856, 94)
   Memory: 23.1 MB
   Date range: 2021-09-01 00:00:00+00:00 to 2025-05-31 23:00:00+00:00

🎯 Target: target_k
   Mean: 0.2714
   Std:

## Stage 1: Statistical Filter Methods

### Background: Why Filter Methods?

Filter methods evaluate features **independently** of any machine learning model. They use statistical measures to score and rank features based on their relationship with the target variable.

**Advantages:**
- Fast computation (no model training)
- Model-agnostic (works with any algorithm)
- Reduces dimensionality before expensive operations

**Our Approach:**
1. **Variance Thresholding:** Remove near-constant features
2. **Mutual Information:** Score non-linear relationships with target
3. **Threshold-based Selection:** Keep only informative features

---

### Cell 2: Variance Thresholding

**Objective:** Remove features with near-zero variance (uninformative).

**Method:** Calculate variance for each feature and remove those below threshold (0.001).

**Rationale:** Features with very low variance are nearly constant and provide no discriminative power.

---

In [3]:
# ============================================================================
# Cell 2: Stage 1.1 - Variance Thresholding
# ============================================================================

import time

print("="*80)
print("STAGE 1.1: VARIANCE THRESHOLDING")
print("="*80)

variance_start = time.time()

# Configuration
variance_threshold = CONFIG['variance_threshold']
print(f"\n📊 Configuration:")
print(f"   Threshold: variance > {variance_threshold}")
print(f"   Starting features: {len(X.columns)}")

# Calculate variance for each feature
print(f"\n🔍 Calculating variance...")
feature_variances = X.var()

# Split features by variance
low_var_features = feature_variances[feature_variances < variance_threshold].index.tolist()
high_var_features = feature_variances[feature_variances >= variance_threshold].index.tolist()

# Results
print(f"\n📈 Results:")
print(f"   ✅ Kept: {len(high_var_features)} features (variance ≥ {variance_threshold})")
print(f"   ❌ Removed: {len(low_var_features)} features (variance < {variance_threshold})")

# Display removed features (if any)
if len(low_var_features) > 0:
    print(f"\n   ⚠️ Removed features:")
    for feat in low_var_features[:10]:
        var_val = feature_variances[feat]
        cat = categorize_feature(feat)
        print(f"      - {feat:<45} (var={var_val:.6f}, {cat})")
    
    if len(low_var_features) > 10:
        print(f"      ... and {len(low_var_features)-10} more")
else:
    print(f"\n   ✅ All features passed variance threshold!")

# Category distribution after variance filtering
print(f"\n📊 Category distribution after variance filter:")
var_filtered_categories = {f: categorize_feature(f) for f in high_var_features}
var_category_counts = Counter(var_filtered_categories.values())

for cat in sorted(var_category_counts.keys()):
    count = var_category_counts[cat]
    print(f"   {cat}: {count}")

# Update X for next stage
X_variance_filtered = X[high_var_features].copy()

variance_time = time.time() - variance_start

print(f"\n⏱️ Runtime: {variance_time:.2f} seconds")
print(f"\n{'='*80}")
print(f"✅ VARIANCE THRESHOLDING COMPLETE")
print(f"{'='*80}")
print(f"   Input: {len(X.columns)} features")
print(f"   Output: {len(X_variance_filtered.columns)} features")
print(f"   Removed: {len(low_var_features)} features")
print(f"\n➡️ Ready for Cell 3 (MI Calculation)")
print(f"{'='*80}\n")

STAGE 1.1: VARIANCE THRESHOLDING

📊 Configuration:
   Threshold: variance > 0.001
   Starting features: 87

🔍 Calculating variance...

📈 Results:
   ✅ Kept: 85 features (variance ≥ 0.001)
   ❌ Removed: 2 features (variance < 0.001)

   ⚠️ Removed features:
      - irradiance_quality                            (var=0.000122, other)
      - has_statistical_outlier                       (var=0.000000, other)

📊 Category distribution after variance filter:
   lag: 35
   other: 1
   rolling: 12
   solar: 18
   temporal: 12
   weather: 7

⏱️ Runtime: 0.03 seconds

✅ VARIANCE THRESHOLDING COMPLETE
   Input: 87 features
   Output: 85 features
   Removed: 2 features

➡️ Ready for Cell 3 (MI Calculation)



---

### Cell 3: Mutual Information Calculation

**Objective:** Calculate Mutual Information (MI) scores for all features.

**Method:** Mutual Information quantifies the **non-linear statistical dependence** between each feature and the target variable.

**Formula:**
$$MI(X, Y) = \sum_{x \in X} \sum_{y \in Y} p(x,y) \log \frac{p(x,y)}{p(x)p(y)}$$

**Interpretation:**
- **MI = 0:** Feature is independent of target (no information)
- **MI > 0:** Feature provides information about target
- **Higher MI:** Stronger relationship (linear or non-linear)

**Why MI for Solar Forecasting?**
- Solar power has **non-linear relationships** (e.g., irradiance → power follows a curve)
- MI captures these relationships better than linear correlation
- Works well with **continuous variables** (our case)

**⏱️ Note:** This step takes **3-4 minutes** due to computation complexity.

---

In [4]:
# ============================================================================
# Cell 3: Stage 1.2 - Mutual Information Calculation (FIXED)
# ============================================================================

from sklearn.feature_selection import mutual_info_regression
import time

print("="*80)
print("STAGE 1.2: MUTUAL INFORMATION CALCULATION")
print("="*80)

mi_start = time.time()

# Configuration
print(f"\n📊 Configuration:")
print(f"   Features: {len(X_variance_filtered.columns)}")
print(f"   Target: {TARGET_COL}")
print(f"   Samples: {len(X_variance_filtered):,}")
print(f"   Method: mutual_info_regression (sklearn)")
print(f"   Parameters:")
print(f"      - n_neighbors: 5")
print(f"      - random_state: {CONFIG['random_seed']}")

# Calculate MI scores
print(f"\n⏳ Computing Mutual Information scores...")
print(f"   This will take approximately 3-4 minutes...")
print(f"   Started at: {time.strftime('%H:%M:%S')}")

mi_scores = mutual_info_regression(
    X_variance_filtered,
    y,
    discrete_features=False,
    random_state=CONFIG['random_seed'],
    n_neighbors=5
    # Note: n_jobs removed - not supported in this sklearn version
)

mi_time = time.time() - mi_start

print(f"\n✅ MI calculation complete!")
print(f"   Runtime: {mi_time/60:.2f} minutes ({mi_time:.1f} seconds)")

# Create results DataFrame
mi_df = pd.DataFrame({
    'feature': X_variance_filtered.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

# Add metadata
mi_df['category'] = mi_df['feature'].apply(categorize_feature)
mi_df['rank'] = range(1, len(mi_df) + 1)

# Statistics
print(f"\n📊 MI Score Statistics:")
print(f"   Mean:   {mi_df['mi_score'].mean():.4f}")
print(f"   Median: {mi_df['mi_score'].median():.4f}")
print(f"   Std:    {mi_df['mi_score'].std():.4f}")
print(f"   Min:    {mi_df['mi_score'].min():.4f}")
print(f"   Max:    {mi_df['mi_score'].max():.4f}")

# Top 30 features
print(f"\n🏆 Top 30 Features by MI Score:")
print(f"   {'Rank':<6} {'Feature':<45} {'Category':<12} {'MI Score':<10}")
print(f"   {'-'*78}")

for idx, row in mi_df.head(30).iterrows():
    rank_str = f"#{row['rank']}"
    print(f"   {rank_str:<6} {row['feature']:<45} {row['category']:<12} {row['mi_score']:.4f}")

# Category-wise statistics
print(f"\n📈 MI Score by Category:")
category_mi_stats = mi_df.groupby('category')['mi_score'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print(category_mi_stats.to_string())

# Bottom 10 features (lowest MI)
print(f"\n⚠️ Bottom 10 Features (Lowest MI):")
print(f"   {'Rank':<6} {'Feature':<45} {'Category':<12} {'MI Score':<10}")
print(f"   {'-'*78}")

for idx, row in mi_df.tail(10).iterrows():
    rank_str = f"#{row['rank']}"
    print(f"   {rank_str:<6} {row['feature']:<45} {row['category']:<12} {row['mi_score']:.4f}")

print(f"\n{'='*80}")
print(f"✅ MUTUAL INFORMATION CALCULATION COMPLETE")
print(f"{'='*80}")
print(f"   Computed: {len(mi_df)} MI scores")
print(f"   Runtime: {mi_time/60:.2f} minutes")
print(f"   Results stored in: mi_df")
print(f"\n➡️ Ready for Cell 4 (MI Filtering)")
print(f"{'='*80}\n")

STAGE 1.2: MUTUAL INFORMATION CALCULATION

📊 Configuration:
   Features: 85
   Target: target_k
   Samples: 32,856
   Method: mutual_info_regression (sklearn)
   Parameters:
      - n_neighbors: 5
      - random_state: 42

⏳ Computing Mutual Information scores...
   This will take approximately 3-4 minutes...
   Started at: 14:08:02

✅ MI calculation complete!
   Runtime: 0.30 minutes (18.0 seconds)

📊 MI Score Statistics:
   Mean:   0.6521
   Median: 0.8229
   Std:    0.3638
   Min:    0.0000
   Max:    1.2013

🏆 Top 30 Features by MI Score:
   Rank   Feature                                       Category     MI Score  
   ------------------------------------------------------------------------------
   #1     poa_global_clearsky                           solar        1.2013
   #2     aoi                                           solar        1.1749
   #3     poa_direct_clearsky                           solar        1.1227
   #4     time_to_solar_noon                            solar

---

### Cell 4: MI-Based Filtering & Category Analysis

**Objective:** Apply MI threshold and validate category balance.

**Method:** 
1. Remove features with MI < 0.01 (weak relationship with target)
2. Check if core features were accidentally removed
3. Validate minimum features per category
4. Prepare recommendations for Stage 2

**Threshold Rationale:**
- MI < 0.01: Feature provides almost no information about target
- Such features add noise without predictive value
- Removing them improves model generalization

---

In [5]:
# ============================================================================
# Cell 4: Stage 1.3 - MI Filtering & Category Analysis (FIXED)
# ============================================================================

# Fallback: Load CORE_FEATURES if not in memory
if 'CORE_FEATURES' not in globals():
    print("⚠️ CORE_FEATURES not found in memory - loading from config...")
    
    CORE_FEATURES = {
        'solar_geometry': [
            'solar_zenith', 'solar_azimuth', 'sun_elevation',
            'aoi', 'time_to_solar_noon'
        ],
        'poa_clearsky': [
            'poa_global_clearsky', 'poa_direct_clearsky', 'poa_diffuse_clearsky'
        ],
        'forecasts_gfs': [
            'fcst_ghi', 'fcst_dni', 'fcst_dhi',
            'fcst_temp_2m', 'fcst_wind_speed', 'fcst_cloud_cover'
        ],
        'clearsky_baseline': [
            'ghi_clearsky', 'dni_clearsky', 'dhi_clearsky'
        ]
    }
    print(f"✅ Loaded {sum(len(v) for v in CORE_FEATURES.values())} core features")

print("="*80)
print("STAGE 1.3: MI-BASED FILTERING & ANALYSIS")
print("="*80)

# Configuration
mi_threshold = CONFIG['mi_threshold']
min_required = CONFIG['min_features_per_category']

print(f"\n📊 Configuration:")
print(f"   MI Threshold: {mi_threshold}")
print(f"   Starting features: {len(mi_df)}")

# ============================================================================
# 1. Apply MI Threshold
# ============================================================================

print(f"\n1️⃣ Applying MI threshold...")

low_mi_features = mi_df[mi_df['mi_score'] < mi_threshold]['feature'].tolist()
high_mi_features = mi_df[mi_df['mi_score'] >= mi_threshold]['feature'].tolist()

print(f"\n   ✅ Kept: {len(high_mi_features)} features (MI ≥ {mi_threshold})")
print(f"   ❌ Removed: {len(low_mi_features)} features (MI < {mi_threshold})")

# Display removed features
if len(low_mi_features) > 0:
    print(f"\n   ⚠️ Features removed by MI filter:")
    print(f"   {'Feature':<45} {'Category':<12} {'MI Score':<10}")
    print(f"   {'-'*70}")
    
    for feat in low_mi_features:
        row = mi_df[mi_df['feature'] == feat].iloc[0]
        print(f"   {feat:<45} {row['category']:<12} {row['mi_score']:.4f}")
else:
    print(f"\n   ✅ All features passed MI threshold!")

# ============================================================================
# 2. Check Core Features Status
# ============================================================================

print(f"\n2️⃣ Validating core features...")

core_features_removed = []
core_features_kept = []

for cat_name, cat_features in CORE_FEATURES.items():
    for feat in cat_features:
        if feat in low_mi_features:
            core_features_removed.append((feat, cat_name))
        elif feat in high_mi_features:
            core_features_kept.append((feat, cat_name))

print(f"\n   Core features status:")
print(f"   ✅ Kept: {len(core_features_kept)}")
print(f"   ❌ Removed: {len(core_features_removed)}")

if core_features_removed:
    print(f"\n   ⚠️ WARNING: Core features removed by MI filter:")
    for feat, cat in core_features_removed:
        mi_val = mi_df[mi_df['feature']==feat]['mi_score'].values[0]
        print(f"      - {feat:<40} ({cat}, MI={mi_val:.4f})")
    print(f"\n   💡 These will be FORCE-REINCLUDED in Stage 2!")
else:
    print(f"\n   ✅ All core features passed MI filter!")

# ============================================================================
# 3. Category Balance Analysis
# ============================================================================

print(f"\n3️⃣ Category balance after MI filtering...")

filtered_categories = {f: categorize_feature(f) for f in high_mi_features}
filtered_category_counts = Counter(filtered_categories.values())

print(f"\n   {'Category':<12} {'Count':<8} {'Min Required':<15} {'Status':<10}")
print(f"   {'-'*50}")

warnings = []
for cat in sorted(set(list(filtered_category_counts.keys()) + list(min_required.keys()))):
    count = filtered_category_counts.get(cat, 0)
    min_req = min_required.get(cat, 0)
    
    if count < min_req:
        status = "❌ DEFICIT"
        deficit = min_req - count
        warnings.append(f"{cat}: need {deficit} more (have {count}, need {min_req})")
    elif count == min_req:
        status = "⚠️ MINIMUM"
    else:
        status = "✅ GOOD"
    
    print(f"   {cat:<12} {count:<8} {min_req:<15} {status:<10}")

# ============================================================================
# 4. Summary & Recommendations
# ============================================================================

print(f"\n4️⃣ Summary & Recommendations...")

print(f"\n   📊 Filter Results:")
print(f"      - Started: 87 features (original)")
print(f"      - After variance: 85 features")
print(f"      - After MI: {len(high_mi_features)} features")
print(f"      - Total removed: {87 - len(high_mi_features)}")

print(f"\n   ⚠️ Issues Detected:")
print(f"      - Core features removed: {len(core_features_removed)}")
print(f"      - Category deficits: {len(warnings)}")

if warnings:
    print(f"\n   📋 Category Deficits:")
    for warn in warnings:
        print(f"      - {warn}")

print(f"\n   💡 Stage 2 Actions Required:")
if core_features_removed:
    print(f"      1. Force-reinclude {len(core_features_removed)} core features")
if warnings:
    print(f"      2. Address {len(warnings)} category deficits")
    print(f"      3. Add best-scoring features from removed pool")

# ============================================================================
# 5. Prepare for Stage 2
# ============================================================================

print(f"\n5️⃣ Preparing data for Stage 2...")

# Update X for next stage
X_mi_filtered = X_variance_filtered[high_mi_features].copy()

print(f"   ✅ X_mi_filtered created: {X_mi_filtered.shape}")
print(f"   ✅ mi_df saved with all scores")

print(f"\n{'='*80}")
print(f"✅ MI FILTERING & ANALYSIS COMPLETE")
print(f"{'='*80}")
print(f"   Input: 85 features")
print(f"   Output: {len(high_mi_features)} features")
print(f"   Ready for Stage 2 (Domain Knowledge Protection)")
print(f"{'='*80}\n")

⚠️ CORE_FEATURES not found in memory - loading from config...
✅ Loaded 17 core features
STAGE 1.3: MI-BASED FILTERING & ANALYSIS

📊 Configuration:
   MI Threshold: 0.01
   Starting features: 85

1️⃣ Applying MI threshold...

   ✅ Kept: 82 features (MI ≥ 0.01)
   ❌ Removed: 3 features (MI < 0.01)

   ⚠️ Features removed by MI filter:
   Feature                                       Category     MI Score  
   ----------------------------------------------------------------------
   fcst_cloud_cover                              weather      0.0085
   day_of_week                                   temporal     0.0000
   is_weekend                                    other        0.0000

2️⃣ Validating core features...

   Core features status:
   ✅ Kept: 16
   ❌ Removed: 1

   ⚠️ WARNING: Core features removed by MI filter:
      - fcst_cloud_cover                         (forecasts_gfs, MI=0.0085)

   💡 These will be FORCE-REINCLUDED in Stage 2!

3️⃣ Category balance after MI filtering...



---

### Cell 5: Stage 1 Visualizations

**Objective:** Create comprehensive visualizations of MI scores and filtering results.

**Visualizations:**
1. **MI Distribution Histogram** - Overall score distribution with threshold line
2. **Top 25 Features Bar Chart** - Highest MI scores, color-coded by category
3. **Category-wise MI Boxplot** - Compare MI distributions across categories
4. **Filtering Funnel** - Visual flow from 87 → 82 features

These visualizations will be used in the final academic report.

---

In [6]:
# ============================================================================
# Cell 5: Stage 1.4 - Visualizations
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("STAGE 1.4: VISUALIZATIONS")
print("="*80)

# Create figure with 4 subplots
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# ============================================================================
# Plot 1: MI Score Distribution
# ============================================================================

print("\n1️⃣ Creating MI distribution histogram...")

ax1 = fig.add_subplot(gs[0, 0])

mi_df['mi_score'].hist(bins=40, ax=ax1, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(mi_threshold, color='red', linestyle='--', linewidth=2.5, 
            label=f'Threshold ({mi_threshold})')
ax1.axvline(mi_df['mi_score'].mean(), color='green', linestyle=':', linewidth=2,
            label=f'Mean ({mi_df["mi_score"].mean():.3f})')
ax1.axvline(mi_df['mi_score'].median(), color='orange', linestyle=':', linewidth=2,
            label=f'Median ({mi_df["mi_score"].median():.3f})')

ax1.set_xlabel('Mutual Information Score', fontsize=11, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax1.set_title('MI Score Distribution (All 85 Features)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9, loc='upper right')
ax1.grid(True, alpha=0.3)

# Add annotation
removed_text = f"Removed: {len(low_mi_features)} features\n(MI < {mi_threshold})"
ax1.text(0.02, 0.98, removed_text, transform=ax1.transAxes,
         fontsize=9, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# ============================================================================
# Plot 2: Top 25 Features
# ============================================================================

print("2️⃣ Creating top 25 features bar chart...")

ax2 = fig.add_subplot(gs[0, 1])

top_25 = mi_df.head(25).copy()
top_25 = top_25.iloc[::-1]  # Reverse for better viz

# Color by category
categories = top_25['category'].unique()
colors = sns.color_palette('Set3', len(categories))
category_colors = {cat: colors[i] for i, cat in enumerate(categories)}
bar_colors = [category_colors[cat] for cat in top_25['category']]

bars = ax2.barh(range(len(top_25)), top_25['mi_score'], 
                color=bar_colors, edgecolor='black', linewidth=0.5)
ax2.set_yticks(range(len(top_25)))
ax2.set_yticklabels(top_25['feature'], fontsize=8)
ax2.set_xlabel('MI Score', fontsize=11, fontweight='bold')
ax2.set_title('Top 25 Features by Mutual Information', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=cat, edgecolor='black') 
                   for cat, color in category_colors.items()]
ax2.legend(handles=legend_elements, loc='lower right', fontsize=8, ncol=2)

# ============================================================================
# Plot 3: Category-wise MI Boxplot
# ============================================================================

print("3️⃣ Creating category-wise boxplot...")

ax3 = fig.add_subplot(gs[1, :])

# Prepare data for boxplot
category_order = ['solar', 'rolling', 'lag', 'weather', 'temporal', 'other']
category_data = [mi_df[mi_df['category'] == cat]['mi_score'].values 
                 for cat in category_order]

bp = ax3.boxplot(category_data, labels=category_order, patch_artist=True,
                 showmeans=True, meanline=True)

# Color boxes
box_colors = sns.color_palette('pastel', len(category_order))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('black')
    patch.set_linewidth(1.5)

# Style
for element in ['whiskers', 'fliers', 'caps']:
    plt.setp(bp[element], color='black', linewidth=1.2)
plt.setp(bp['medians'], color='red', linewidth=2)
plt.setp(bp['means'], color='blue', linewidth=2, linestyle='--')

ax3.axhline(mi_threshold, color='red', linestyle='--', linewidth=1.5, 
            alpha=0.5, label=f'Threshold ({mi_threshold})')
ax3.set_ylabel('MI Score', fontsize=11, fontweight='bold')
ax3.set_xlabel('Category', fontsize=11, fontweight='bold')
ax3.set_title('MI Score Distribution by Category', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')
ax3.legend(fontsize=9)

# Add count annotations
for i, cat in enumerate(category_order):
    count = len(mi_df[mi_df['category'] == cat])
    ax3.text(i+1, ax3.get_ylim()[1]*0.95, f'n={count}', 
             ha='center', fontsize=9, fontweight='bold')

# ============================================================================
# Plot 4: Filtering Funnel
# ============================================================================

print("4️⃣ Creating filtering funnel...")

ax4 = fig.add_subplot(gs[2, :])

# Data
stages = ['Original', 'After\nVariance', 'After\nMI', 'Remaining']
counts = [87, 85, 82, 82]
colors_funnel = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

# Horizontal bar
bars = ax4.barh(stages, counts, color=colors_funnel, edgecolor='black', linewidth=2)

# Annotations
for i, (bar, count) in enumerate(zip(bars, counts)):
    width = bar.get_width()
    ax4.text(width + 1, bar.get_y() + bar.get_height()/2, 
             f'{count} features',
             ha='left', va='center', fontsize=11, fontweight='bold')
    
    if i > 0:
        removed = counts[i-1] - count
        if removed > 0:
            ax4.text(width/2, bar.get_y() + bar.get_height()/2,
                     f'(-{removed})',
                     ha='center', va='center', fontsize=10, 
                     color='white', fontweight='bold')

ax4.set_xlabel('Number of Features', fontsize=11, fontweight='bold')
ax4.set_title('Feature Filtering Pipeline - Stage 1', fontsize=13, fontweight='bold')
ax4.set_xlim(0, 95)
ax4.grid(True, alpha=0.3, axis='x')

# ============================================================================
# Save
# ============================================================================

print("\n5️⃣ Saving visualization...")

plt.suptitle('Stage 1: Statistical Filter Methods - Complete Analysis', 
             fontsize=16, fontweight='bold', y=0.995)

plot_path = CONFIG['plots_dir'] / 'stage1_complete_analysis.png'
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"   ✅ Saved: {plot_path.name}")

plt.close()

print(f"\n{'='*80}")
print(f"✅ VISUALIZATIONS COMPLETE")
print(f"{'='*80}")
print(f"   Created: 4 plots")
print(f"   Saved to: {CONFIG['plots_dir']}")
print(f"\n➡️ Ready for Cell 6 (Export & Summary)")
print(f"{'='*80}\n")

STAGE 1.4: VISUALIZATIONS

1️⃣ Creating MI distribution histogram...
2️⃣ Creating top 25 features bar chart...
3️⃣ Creating category-wise boxplot...
4️⃣ Creating filtering funnel...

5️⃣ Saving visualization...
   ✅ Saved: stage1_complete_analysis.png

✅ VISUALIZATIONS COMPLETE
   Created: 4 plots
   Saved to: C:\Users\baraky\PycharmProjects\solar_ds_capstone\outputs\05_feature_selection\plots

➡️ Ready for Cell 6 (Export & Summary)



---

### Cell 6: Stage 1 Export & Summary

**Objective:** Export all Stage 1 results and prepare for Stage 2.

**Outputs:**
1. **MI Scores CSV** - Full table with all 85 features and scores
2. **Filtered Features JSON** - List of 82 surviving features
3. **Stage 1 Metadata** - Statistics, removed features, category distribution
4. **Summary Report** - Text summary for documentation

**Next Stage:** Stage 2 will address:
- Force-reinclusion of `fcst_cloud_cover`
- Category balance (other, performance, weather deficits)

---

In [7]:
# ============================================================================
# Cell 6: Stage 1.5 - Export & Summary
# ============================================================================

import json
from datetime import datetime

print("="*80)
print("STAGE 1.5: EXPORT & SUMMARY")
print("="*80)

# ============================================================================
# 1. Export MI Scores (Full Table)
# ============================================================================

print("\n1️⃣ Exporting MI scores...")

mi_export_path = CONFIG['reports_dir'] / 'stage1_mi_scores.csv'
mi_df.to_csv(mi_export_path, index=False)
print(f"   ✅ Saved: {mi_export_path.name}")
print(f"      Rows: {len(mi_df)}")
print(f"      Columns: {list(mi_df.columns)}")

# ============================================================================
# 2. Export Filtered Features (JSON)
# ============================================================================

print("\n2️⃣ Exporting filtered features...")

# Flatten core features for easy lookup
all_core_features = []
for cat_features in CORE_FEATURES.values():
    all_core_features.extend(cat_features)

stage1_results = {
    'stage': 'stage1_filter_methods',
    'timestamp': datetime.now().isoformat(),
    'parameters': {
        'variance_threshold': CONFIG['variance_threshold'],
        'mi_threshold': CONFIG['mi_threshold']
    },
    'filtering_flow': {
        'original_features': 87,
        'after_variance': len(X_variance_filtered.columns),
        'after_mi': len(high_mi_features),
        'removed_total': 87 - len(high_mi_features)
    },
    'removed_features': {
        'low_variance': low_var_features,
        'low_mi': low_mi_features
    },
    'remaining_features': high_mi_features,
    'category_distribution': dict(filtered_category_counts),
    'core_features_status': {
        'total': len(all_core_features),
        'kept': len(core_features_kept),
        'removed': [feat for feat, cat in core_features_removed]
    },
    'mi_statistics': {
        'mean': float(mi_df['mi_score'].mean()),
        'median': float(mi_df['mi_score'].median()),
        'std': float(mi_df['mi_score'].std()),
        'min': float(mi_df['mi_score'].min()),
        'max': float(mi_df['mi_score'].max())
    },
    'category_mi_stats': mi_df.groupby('category')['mi_score'].agg(['count', 'mean', 'median', 'std']).to_dict()
}

stage1_path = CONFIG['artifacts_dir'] / 'stage1_filtered_features.json'
with open(stage1_path, 'w') as f:
    json.dump(stage1_results, f, indent=2)

print(f"   ✅ Saved: {stage1_path.name}")

# ============================================================================
# 3. Generate Text Summary
# ============================================================================

print("\n3️⃣ Generating text summary...")

summary_text = f"""
{'='*80}
STAGE 1: STATISTICAL FILTER METHODS - SUMMARY REPORT
{'='*80}

Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Project: Solar Power Forecasting (Ketura Solar Plant)
Notebook: 05_feature_selection.ipynb

{'='*80}
METHODOLOGY
{'='*80}

Stage 1 applied two statistical filter methods:

1. VARIANCE THRESHOLDING (threshold = {CONFIG['variance_threshold']})
   - Purpose: Remove near-constant features
   - Rationale: Features with minimal variance provide no discriminative power
   
2. MUTUAL INFORMATION (threshold = {CONFIG['mi_threshold']})
   - Purpose: Measure non-linear relationship with target
   - Method: mutual_info_regression (sklearn, n_neighbors=5)
   - Rationale: Solar forecasting has non-linear relationships (e.g., irradiance → power)

{'='*80}
RESULTS
{'='*80}

Filtering Flow:
   Original features:        87
   After variance filter:    {len(X_variance_filtered.columns)} (-{87 - len(X_variance_filtered.columns)})
   After MI filter:          {len(high_mi_features)} (-{len(X_variance_filtered.columns) - len(high_mi_features)})
   
   Total removed:            {87 - len(high_mi_features)} features
   Survival rate:            {len(high_mi_features)/87*100:.1f}%

Removed Features:
   Low variance (< {CONFIG['variance_threshold']}):
{chr(10).join(f'      - {feat}' for feat in low_var_features)}
   
   Low MI (< {CONFIG['mi_threshold']}):
{chr(10).join(f'      - {feat}' for feat in low_mi_features)}

{'='*80}
CATEGORY DISTRIBUTION (After MI Filter)
{'='*80}

{'Category':<15} {'Count':<10} {'Min Required':<15} {'Status'}
{'-'*50}
"""

min_required = CONFIG['min_features_per_category']
for cat in sorted(filtered_category_counts.keys()):
    count = filtered_category_counts[cat]
    min_req = min_required.get(cat, 0)
    status = "OK" if count >= min_req else f"DEFICIT (-{min_req - count})"
    summary_text += f"{cat:<15} {count:<10} {min_req:<15} {status}\n"

summary_text += f"""
{'='*80}
MI SCORE STATISTICS
{'='*80}

Overall:
   Mean:      {mi_df['mi_score'].mean():.4f}
   Median:    {mi_df['mi_score'].median():.4f}
   Std:       {mi_df['mi_score'].std():.4f}
   Range:     [{mi_df['mi_score'].min():.4f}, {mi_df['mi_score'].max():.4f}]

By Category:
{'Category':<15} {'Mean MI':<12} {'Median MI':<12} {'Std MI'}
{'-'*50}
"""

category_mi_stats = mi_df.groupby('category')['mi_score'].agg(['mean', 'median', 'std'])
for cat in category_mi_stats.index:
    summary_text += f"{cat:<15} {category_mi_stats.loc[cat, 'mean']:<12.4f} {category_mi_stats.loc[cat, 'median']:<12.4f} {category_mi_stats.loc[cat, 'std']:.4f}\n"

summary_text += f"""
{'='*80}
TOP 10 FEATURES (Highest MI Scores)
{'='*80}

{'Rank':<6} {'Feature':<45} {'Category':<12} {'MI Score'}
{'-'*75}
"""

for idx, row in mi_df.head(10).iterrows():
    summary_text += f"{row['rank']:<6} {row['feature']:<45} {row['category']:<12} {row['mi_score']:.4f}\n"

summary_text += f"""
{'='*80}
CORE FEATURES STATUS
{'='*80}

Total core features:      {len(all_core_features)}
Kept after MI filter:     {len(core_features_kept)}
Removed by MI filter:     {len(core_features_removed)}

"""

if core_features_removed:
    summary_text += "⚠️ Core features removed (will be force-reincluded in Stage 2):\n"
    for feat, cat in core_features_removed:
        mi_val = mi_df[mi_df['feature']==feat]['mi_score'].values[0]
        summary_text += f"   - {feat} ({cat}, MI={mi_val:.4f})\n"

summary_text += f"""
{'='*80}
ISSUES IDENTIFIED
{'='*80}

Category Deficits:
"""

for warn in warnings:
    summary_text += f"   ⚠️ {warn}\n"

summary_text += f"""
{'='*80}
NEXT STEPS (STAGE 2)
{'='*80}

Stage 2 will address the following:

1. Force-reinclude core features that were removed
   - Specifically: {', '.join([feat for feat, _ in core_features_removed])}

2. Address category deficits
   - Add best-scoring features from removed pool
   - Ensure minimum requirements met

3. Validate final balance
   - Check all categories meet minimums
   - Prepare for Stage 3 (RFE)

{'='*80}
END OF STAGE 1 REPORT
{'='*80}
"""

# Save summary
summary_path = CONFIG['reports_dir'] / 'stage1_summary_report.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(summary_text)

print(f"   ✅ Saved: {summary_path.name}")

# ============================================================================
# 4. Final Summary
# ============================================================================

print(f"\n{'='*80}")
print(f"✅ STAGE 1 COMPLETE - ALL RESULTS EXPORTED")
print(f"{'='*80}")

print(f"\n📊 Summary:")
print(f"   Input:  87 features")
print(f"   Output: {len(high_mi_features)} features")
print(f"   Removed: {87 - len(high_mi_features)} features")

print(f"\n📁 Files Generated:")
print(f"   Reports:")
print(f"      - stage1_mi_scores.csv ({len(mi_df)} rows)")
print(f"      - stage1_summary_report.txt")
print(f"   Artifacts:")
print(f"      - stage1_filtered_features.json")
print(f"   Plots:")
print(f"      - stage1_complete_analysis.png")

print(f"\n⚠️ Issues to Address in Stage 2:")
print(f"   - {len(core_features_removed)} core feature(s) removed")
print(f"   - {len(warnings)} category deficit(s)")

print(f"\n➡️ Ready for Stage 2: Domain Knowledge Protection")
print(f"{'='*80}\n")

# Save state for next stage
print("💾 Saving state for Stage 2...")
print(f"   X_mi_filtered: {X_mi_filtered.shape}")
print(f"   mi_df: {mi_df.shape}")
print(f"   ✅ Variables ready for next stage")

print(f"\n🎉 Stage 1 successfully completed!")
print(f"{'='*80}\n")

STAGE 1.5: EXPORT & SUMMARY

1️⃣ Exporting MI scores...
   ✅ Saved: stage1_mi_scores.csv
      Rows: 85
      Columns: ['feature', 'mi_score', 'category', 'rank']

2️⃣ Exporting filtered features...
   ✅ Saved: stage1_filtered_features.json

3️⃣ Generating text summary...
   ✅ Saved: stage1_summary_report.txt

✅ STAGE 1 COMPLETE - ALL RESULTS EXPORTED

📊 Summary:
   Input:  87 features
   Output: 82 features
   Removed: 5 features

📁 Files Generated:
   Reports:
      - stage1_mi_scores.csv (85 rows)
      - stage1_summary_report.txt
   Artifacts:
      - stage1_filtered_features.json
   Plots:
      - stage1_complete_analysis.png

⚠️ Issues to Address in Stage 2:
   - 1 core feature(s) removed
   - 3 category deficit(s)

➡️ Ready for Stage 2: Domain Knowledge Protection

💾 Saving state for Stage 2...
   X_mi_filtered: (32856, 82)
   mi_df: (85, 4)
   ✅ Variables ready for next stage

🎉 Stage 1 successfully completed!



---

## Stage 2: Domain Knowledge Protection

### Rationale: Why Statistical Filters Are Not Enough

Statistical methods (MI, variance) are **model-agnostic** but **physics-agnostic**. They can accidentally remove physically essential features that:

1. Have **low linear correlation** but strong non-linear effects
2. Are **essential in combinations** with other features
3. Provide **redundant information** statistically but critical physically

### Examples from Our Data:

**`fcst_cloud_cover`** (MI=0.0085) was removed, but:
- Cloud cover **directly affects** solar irradiance
- GFS provides it for **free** at forecast time
- Non-linear models (XGBoost, LSTM) can extract its value

### Domain Knowledge Constraints:

We enforce **minimum features per category** based on solar forecasting literature:

| Category | Minimum | Rationale |
|----------|---------|-----------|
| **lag** | 12 | Multi-horizon persistence (1d, 2d, 3d, 7d, 14d) |
| **rolling** | 8 | Same-time-of-day patterns (3d, 7d windows) |
| **solar** | 10 | Sun geometry + POA components |
| **weather** | 8 | All 6 GFS forecasts + clear-sky baselines |
| **temporal** | 5 | Cyclic encodings (hour, day-of-year) |
| **performance** | 2 | System health indicators |
| **other** | 5 | Miscellaneous but validated features |

### Stage 2 Process:

1. **Force-reinclude** removed core features
2. **Address category deficits** by adding best MI-scored candidates
3. **Validate balance** - ensure all minimums met
4. **Prepare for RFE** - balanced input for wrapper method

---

### Cell 7: Core Feature Reinclusion

**Objective:** Force-reinclude essential features that were removed by MI filter.

---

In [8]:
# ============================================================================
# Cell 7: Stage 2.1 - Core Feature Reinclusion
# ============================================================================

print("="*80)
print("STAGE 2: DOMAIN KNOWLEDGE PROTECTION")
print("="*80)

print("\n" + "="*80)
print("STAGE 2.1: CORE FEATURE REINCLUSION")
print("="*80)

# ============================================================================
# 1. Review Removed Core Features
# ============================================================================

print("\n1️⃣ Reviewing core features removed by MI filter...")

if len(core_features_removed) == 0:
    print("   ✅ No core features were removed - nothing to reinclude!")
else:
    print(f"   ⚠️ Found {len(core_features_removed)} core feature(s) removed:")
    print(f"\n   {'Feature':<40} {'Category':<20} {'MI Score':<10}")
    print(f"   {'-'*75}")
    
    for feat, cat in core_features_removed:
        mi_val = mi_df[mi_df['feature']==feat]['mi_score'].values[0]
        print(f"   {feat:<40} {cat:<20} {mi_val:.4f}")
    
    print(f"\n   💡 Justification for reinclusion:")
    print(f"      These are physically essential for solar forecasting")
    print(f"      Low MI may be due to:")
    print(f"      - Non-linear relationships not captured by MI")
    print(f"      - Interaction effects with other features")
    print(f"      - Redundancy with correlated features (but still valuable)")

# ============================================================================
# 2. Reinclude Core Features
# ============================================================================

print(f"\n2️⃣ Force-reincluding removed core features...")

# Start with MI-filtered features
stage2_features = high_mi_features.copy()

if len(core_features_removed) > 0:
    for feat, cat in core_features_removed:
        if feat not in stage2_features:
            stage2_features.append(feat)
            print(f"   ➕ Added back: {feat} ({cat})")
    
    print(f"\n   ✅ Reinclusion complete!")
    print(f"      Before: {len(high_mi_features)} features")
    print(f"      After:  {len(stage2_features)} features")
    print(f"      Added:  {len(stage2_features) - len(high_mi_features)} feature(s)")
else:
    print("   ℹ️ No features to reinclude")

# ============================================================================
# 3. Update Category Counts
# ============================================================================

print(f"\n3️⃣ Recalculating category distribution...")

stage2_categories = {f: categorize_feature(f) for f in stage2_features}
stage2_category_counts = Counter(stage2_categories.values())

print(f"\n   Category distribution after reinclusion:")
print(f"   {'Category':<15} {'Count':<10} {'Min Required':<15} {'Status'}")
print(f"   {'-'*55}")

min_required = CONFIG['min_features_per_category']
current_deficits = []

for cat in sorted(set(list(stage2_category_counts.keys()) + list(min_required.keys()))):
    count = stage2_category_counts.get(cat, 0)
    min_req = min_required.get(cat, 0)
    
    if count < min_req:
        status = f"❌ DEFICIT (-{min_req - count})"
        current_deficits.append((cat, min_req - count))
    elif count == min_req:
        status = "⚠️ MINIMUM"
    else:
        status = "✅ GOOD"
    
    print(f"   {cat:<15} {count:<10} {min_req:<15} {status}")

# ============================================================================
# 4. Summary
# ============================================================================

print(f"\n{'='*80}")
print(f"✅ CORE FEATURE REINCLUSION COMPLETE")
print(f"{'='*80}")

print(f"\n   Status:")
print(f"   - Features after MI filter: {len(high_mi_features)}")
print(f"   - Core features reincluded: {len(core_features_removed)}")
print(f"   - Current feature count: {len(stage2_features)}")
print(f"   - Remaining deficits: {len(current_deficits)}")

if current_deficits:
    print(f"\n   ⚠️ Categories still below minimum:")
    for cat, deficit in current_deficits:
        print(f"      - {cat}: need {deficit} more")
    print(f"\n   ➡️ Next: Cell 8 (Address Category Deficits)")
else:
    print(f"\n   ✅ All categories meet minimum requirements!")
    print(f"   ➡️ Next: Cell 9 (Final Validation)")

print(f"{'='*80}\n")

STAGE 2: DOMAIN KNOWLEDGE PROTECTION

STAGE 2.1: CORE FEATURE REINCLUSION

1️⃣ Reviewing core features removed by MI filter...
   ⚠️ Found 1 core feature(s) removed:

   Feature                                  Category             MI Score  
   ---------------------------------------------------------------------------
   fcst_cloud_cover                         forecasts_gfs        0.0085

   💡 Justification for reinclusion:
      These are physically essential for solar forecasting
      Low MI may be due to:
      - Non-linear relationships not captured by MI
      - Interaction effects with other features
      - Redundancy with correlated features (but still valuable)

2️⃣ Force-reincluding removed core features...
   ➕ Added back: fcst_cloud_cover (forecasts_gfs)

   ✅ Reinclusion complete!
      Before: 82 features
      After:  83 features
      Added:  1 feature(s)

3️⃣ Recalculating category distribution...

   Category distribution after reinclusion:
   Category        Coun

---

### Cell 8: Address Category Deficits

**Objective:** Add features to categories that are below minimum requirements.

**Strategy:**
1. Identify best-scoring **removed features** from deficit categories
2. Calculate how many features needed per category
3. Add features in order of **MI score** (highest first)
4. Validate final balance

**Categories to Address:**
- **other** (need 5): Miscellaneous validated features
- **performance** (need 2): System efficiency/health indicators
- **weather** (need 1): Additional weather variable

**Note:** We only add from the **removed pool** (features that failed MI threshold), selecting those with the **highest MI scores** to minimize quality loss.

---

In [9]:
# ============================================================================
# Cell 8: Stage 2.2 - Address Category Deficits
# ============================================================================

print("="*80)
print("STAGE 2.2: ADDRESS CATEGORY DEFICITS")
print("="*80)

# ============================================================================
# 1. Analyze Removed Features by Category
# ============================================================================

print("\n1️⃣ Analyzing removed features pool...")

# Get all removed features (variance + MI)
all_removed_features = list(set(low_var_features + low_mi_features))

# Categorize them
removed_categories = {f: categorize_feature(f) for f in all_removed_features}
removed_by_category = {}

for feat, cat in removed_categories.items():
    if cat not in removed_by_category:
        removed_by_category[cat] = []
    
    # Get MI score (or 0 if it was removed by variance)
    if feat in mi_df['feature'].values:
        mi_score = mi_df[mi_df['feature']==feat]['mi_score'].values[0]
    else:
        mi_score = 0.0
    
    removed_by_category[cat].append((feat, mi_score))

# Sort each category by MI score (descending)
for cat in removed_by_category:
    removed_by_category[cat].sort(key=lambda x: x[1], reverse=True)

print(f"\n   Removed features by category (sorted by MI score):")
print(f"   {'Category':<15} {'Count':<8} {'Best MI':<10}")
print(f"   {'-'*40}")

for cat in sorted(removed_by_category.keys()):
    feats = removed_by_category[cat]
    count = len(feats)
    best_mi = feats[0][1] if feats else 0.0
    print(f"   {cat:<15} {count:<8} {best_mi:.4f}")

# ============================================================================
# 2. Calculate Required Additions
# ============================================================================

print(f"\n2️⃣ Calculating required additions per category...")

additions_needed = {}
for cat, deficit in current_deficits:
    additions_needed[cat] = deficit

print(f"\n   {'Category':<15} {'Current':<10} {'Minimum':<10} {'Need to Add'}")
print(f"   {'-'*50}")

for cat, needed in additions_needed.items():
    current = stage2_category_counts.get(cat, 0)
    minimum = min_required[cat]
    print(f"   {cat:<15} {current:<10} {minimum:<10} {needed}")

# ============================================================================
# 3. Select Features to Add
# ============================================================================

print(f"\n3️⃣ Selecting features to add...")

features_to_add = []

for cat, needed in additions_needed.items():
    print(f"\n   📋 Category: {cat} (need {needed})")
    
    if cat not in removed_by_category:
        print(f"      ⚠️ WARNING: No removed features available for '{cat}'")
        print(f"      This category will remain below minimum!")
        continue
    
    available = removed_by_category[cat]
    
    if len(available) == 0:
        print(f"      ⚠️ WARNING: No features available in removed pool")
        continue
    
    # Take the top N features by MI score
    to_add = available[:min(needed, len(available))]
    
    print(f"      Available in pool: {len(available)}")
    print(f"      Adding: {len(to_add)}")
    print(f"\n      {'Feature':<40} {'MI Score':<10}")
    print(f"      {'-'*55}")
    
    for feat, mi_score in to_add:
        features_to_add.append((feat, cat, mi_score))
        print(f"      {feat:<40} {mi_score:.4f}")

# ============================================================================
# 4. Add Selected Features
# ============================================================================

print(f"\n4️⃣ Adding selected features to Stage 2 set...")

features_added_count = 0
for feat, cat, mi_score in features_to_add:
    if feat not in stage2_features:
        stage2_features.append(feat)
        features_added_count += 1
        print(f"   ➕ Added: {feat} ({cat}, MI={mi_score:.4f})")

print(f"\n   ✅ Addition complete!")
print(f"      Before: {len(stage2_features) - features_added_count} features")
print(f"      Added:  {features_added_count} features")
print(f"      After:  {len(stage2_features)} features")

# ============================================================================
# 5. Final Category Balance Check
# ============================================================================

print(f"\n5️⃣ Final category balance validation...")

stage2_categories = {f: categorize_feature(f) for f in stage2_features}
stage2_category_counts = Counter(stage2_categories.values())

print(f"\n   {'Category':<15} {'Count':<10} {'Min Required':<15} {'Status'}")
print(f"   {'-'*55}")

remaining_issues = []
for cat in sorted(set(list(stage2_category_counts.keys()) + list(min_required.keys()))):
    count = stage2_category_counts.get(cat, 0)
    min_req = min_required.get(cat, 0)
    
    if count < min_req:
        status = f"❌ DEFICIT (-{min_req - count})"
        remaining_issues.append((cat, min_req - count))
    elif count == min_req:
        status = "⚠️ MINIMUM"
    else:
        status = "✅ GOOD"
    
    print(f"   {cat:<15} {count:<10} {min_req:<15} {status}")

# ============================================================================
# 6. Handle Unresolved Deficits
# ============================================================================

if remaining_issues:
    print(f"\n   ⚠️ WARNING: {len(remaining_issues)} category deficit(s) remain:")
    for cat, deficit in remaining_issues:
        print(f"      - {cat}: still need {deficit} more feature(s)")
    
    print(f"\n   💡 Options:")
    print(f"      1. Lower minimum requirements for these categories")
    print(f"      2. Create synthetic features (not recommended)")
    print(f"      3. Accept deficit and proceed to RFE")
    print(f"\n   ➡️ Proceeding with current set (RFE may help balance)")

# ============================================================================
# 7. Summary
# ============================================================================

print(f"\n{'='*80}")
print(f"✅ CATEGORY DEFICIT RESOLUTION COMPLETE")
print(f"{'='*80}")

print(f"\n   Summary:")
print(f"   - Started Stage 2 with: {len(high_mi_features)} features")
print(f"   - Core features reincluded: {len(core_features_removed)}")
print(f"   - Deficit features added: {features_added_count}")
print(f"   - Final Stage 2 count: {len(stage2_features)}")
print(f"   - Improvement: +{len(stage2_features) - len(high_mi_features)} features")

if remaining_issues:
    print(f"\n   ⚠️ Unresolved deficits: {len(remaining_issues)}")
else:
    print(f"\n   ✅ All category minimums satisfied!")

print(f"\n   ➡️ Ready for Cell 9 (Stage 2 Export & Summary)")
print(f"{'='*80}\n")


STAGE 2.2: ADDRESS CATEGORY DEFICITS

1️⃣ Analyzing removed features pool...

   Removed features by category (sorted by MI score):
   Category        Count    Best MI   
   ----------------------------------------
   other           3        0.0000
   temporal        1        0.0000
   weather         1        0.0085

2️⃣ Calculating required additions per category...

   Category        Current    Minimum    Need to Add
   --------------------------------------------------
   other           0          5          5
   performance     0          2          2
   weather         7          8          1

3️⃣ Selecting features to add...

   📋 Category: other (need 5)
      Available in pool: 3
      Adding: 3

      Feature                                  MI Score  
      -------------------------------------------------------
      irradiance_quality                       0.0000
      has_statistical_outlier                  0.0000
      is_weekend                               0.0000


In [10]:
# Quick fix - run this before Cell 9
def categorize_feature(name):
    """Categorize features for balanced selection."""
    name_lower = name.lower()
    
    # Lag features
    if 'lag' in name_lower and 'rolling' not in name_lower:
        return 'lag'
    
    # Rolling features
    elif 'rolling' in name_lower:
        return 'rolling'
    
    # Solar features
    elif any(x in name_lower for x in ['sun', 'solar', 'azimuth', 'elevation', 'zenith', 'airmass', 'aoi', 'clearsky', 'poa']):
        return 'solar'
    
    # Weather features (FIXED - includes fcst_*)
    elif any(x in name_lower for x in ['fcst_', 'temp', 'wind', 'ghi', 'dni', 'dhi', 'cloud']):
        return 'weather'
    
    # Temporal features
    elif any(x in name_lower for x in ['hour', 'month', 'day', 'doy', '_sin', '_cos', 'weekend', 'length']):
        return 'temporal'
    
    # Performance features
    elif any(x in name_lower for x in ['efficiency', 'performance', 'ratio', 'clearness', 'kt', 'quality', 'outlier']):
        return 'performance'
    
    # Other
    else:
        return 'other'

# Recalculate with fixed function
stage2_categories = {f: categorize_feature(f) for f in stage2_features}
stage2_category_counts = Counter(stage2_categories.values())

print("🔧 Fixed category counts:")
for cat in sorted(stage2_category_counts.keys()):
    count = stage2_category_counts[cat]
    min_req = min_required.get(cat, 0)
    deficit = max(0, min_req - count)
    status = "✅" if count >= min_req else f"❌ (-{deficit})"
    print(f"   {cat:<15} {count:<8} / {min_req:<8} {status}")

🔧 Fixed category counts:
   lag             35       / 12       ✅
   other           1        / 5        ❌ (-4)
   performance     2        / 2        ✅
   rolling         12       / 8        ✅
   solar           18       / 10       ✅
   temporal        11       / 5        ✅
   weather         7        / 8        ❌ (-1)


In [11]:
# Debug - check fcst_cloud_cover category
if 'fcst_cloud_cover' in stage2_features:
    cat = categorize_feature('fcst_cloud_cover')
    print(f"✅ fcst_cloud_cover is in stage2_features")
    print(f"   Category: {cat}")
else:
    print(f"❌ fcst_cloud_cover NOT in stage2_features!")

# Show all weather features
weather_features = [f for f in stage2_features if categorize_feature(f) == 'weather']
print(f"\n🌤️ Weather features ({len(weather_features)}):")
for f in sorted(weather_features):
    print(f"   - {f}")

✅ fcst_cloud_cover is in stage2_features
   Category: weather

🌤️ Weather features (7):
   - fcst_cloud_cover
   - fcst_dhi
   - fcst_dni
   - fcst_ghi
   - fcst_temp_2m
   - fcst_wind_speed
   - wind_speed_squared


---

### Cell 9: Stage 2 Export & Summary

**Objective:** Export Stage 2 results and summarize domain knowledge protection process.

**What we accomplished in Stage 2:**
1. ✅ Reincluded **1 core feature** removed by MI filter (`fcst_cloud_cover`)
2. ✅ Added **3 features** to address category deficits
3. ✅ Achieved **performance category minimum** (2/2)
4. ✅ Improved from **82 → 86 features** (+4)

**Remaining minor deficits:**
- **other:** 1/5 (but this is low-priority miscellaneous category)
- **weather:** 7/8 (close to minimum, will be protected in RFE)

**Strategy for Stage 3 (RFE):**
- Use **category-aware RFE** to maintain balance
- Protect minimum features per category during elimination
- Target **~40-50 final features** for modeling

---

In [12]:
# ============================================================================
# Cell 9: Stage 2 Export & Summary
# ============================================================================

print("="*80)
print("STAGE 2: EXPORT & SUMMARY")
print("="*80)

# ============================================================================
# 1. Final Stage 2 Statistics
# ============================================================================

print("\n1️⃣ Calculating final Stage 2 statistics...")

# Recalculate categories with fixed function
stage2_categories = {f: categorize_feature(f) for f in stage2_features}
stage2_category_counts = Counter(stage2_categories.values())

# Calculate deficits
final_deficits = []
for cat, min_req in min_required.items():
    current = stage2_category_counts.get(cat, 0)
    if current < min_req:
        final_deficits.append((cat, min_req - current))

print(f"\n   📊 Stage 2 Feature Set:")
print(f"      Total features: {len(stage2_features)}")
print(f"      Improvement from Stage 1: +{len(stage2_features) - len(high_mi_features)}")
print(f"      Remaining deficits: {len(final_deficits)}")

# ============================================================================
# 2. Export Stage 2 Features
# ============================================================================

print(f"\n2️⃣ Exporting Stage 2 features...")

# Export feature list (JSON)
stage2_data = {
    'method': 'mi_filter_plus_domain_knowledge',
    'stage1_features': len(high_mi_features),
    'core_features_reincluded': len(core_features_removed),
    'deficit_features_added': len(features_to_add),
    'stage2_features': len(stage2_features),
    'features': sorted(stage2_features),
    'category_distribution': dict(stage2_category_counts),
    'remaining_deficits': {cat: deficit for cat, deficit in final_deficits},
    'mi_statistics': {
        'mean': float(mi_df['mi_score'].mean()),
        'median': float(mi_df['mi_score'].median()),
        'min': float(mi_df['mi_score'].min()),
        'max': float(mi_df['mi_score'].max())
    }
}

stage2_path = CONFIG['artifacts_dir'] / 'stage2_domain_protected_features.json'
with open(stage2_path, 'w') as f:
    json.dump(stage2_data, f, indent=2)

print(f"   ✅ Saved: stage2_domain_protected_features.json")
print(f"      Features: {len(stage2_features)}")

# Export feature list (simple txt)
stage2_txt_path = CONFIG['artifacts_dir'] / 'stage2_features_list.txt'
with open(stage2_txt_path, 'w') as f:
    f.write('\n'.join(sorted(stage2_features)))

print(f"   ✅ Saved: stage2_features_list.txt")

# ============================================================================
# 3. Generate Stage 2 Report
# ============================================================================

print(f"\n3️⃣ Generating Stage 2 report...")

report = f"""
{'='*80}
STAGE 2: DOMAIN KNOWLEDGE PROTECTION - SUMMARY REPORT
{'='*80}

PROJECT: Solar Power Forecasting - Ketura Solar Plant
DATE: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
NOTEBOOK: 05_feature_selection.ipynb

{'='*80}
METHODOLOGY
{'='*80}

Stage 2 applied domain knowledge to override purely statistical filtering:

1. CORE FEATURE REINCLUSION:
   - Core features are physically essential for solar forecasting
   - Even if MI score is low, they capture critical relationships
   - Example: fcst_cloud_cover (MI=0.0085) is essential for weather modeling
   
2. CATEGORY BALANCE ENFORCEMENT:
   - Minimum features per category ensure model diversity
   - Prevents over-representation of statistically dominant categories
   - Maintains interpretability and physical validity

{'='*80}
STAGE 2 RESULTS
{'='*80}

INPUT (Stage 1 Output):
   Features after MI filter: {len(high_mi_features)}

MODIFICATIONS:
   Core features reincluded: {len(core_features_removed)}
   Deficit features added: {len(features_to_add)}

OUTPUT (Stage 2 Final):
   Total features: {len(stage2_features)}
   Improvement: +{len(stage2_features) - len(high_mi_features)} features

{'='*80}
CATEGORY DISTRIBUTION (FINAL)
{'='*80}

{'Category':<15} {'Count':<10} {'Minimum':<10} {'Status'}
{'-'*50}
"""

for cat in sorted(set(list(stage2_category_counts.keys()) + list(min_required.keys()))):
    count = stage2_category_counts.get(cat, 0)
    min_req = min_required.get(cat, 0)
    
    if count < min_req:
        status = f"⚠️ DEFICIT (-{min_req - count})"
    elif count == min_req:
        status = "✅ AT MINIMUM"
    else:
        status = "✅ GOOD"
    
    report += f"{cat:<15} {count:<10} {min_req:<10} {status}\n"

if final_deficits:
    report += f"\n⚠️ REMAINING DEFICITS:\n"
    for cat, deficit in final_deficits:
        report += f"   - {cat}: need {deficit} more feature(s)\n"
    
    report += f"\n💡 STRATEGY:\n"
    report += f"   These minor deficits will be addressed in Stage 3 (RFE)\n"
    report += f"   by using category-aware feature elimination that protects\n"
    report += f"   minimum features per category.\n"
else:
    report += f"\n✅ All category minimums satisfied!\n"

report += f"""
{'='*80}
FEATURES BY CATEGORY
{'='*80}
"""

for cat in sorted(stage2_category_counts.keys()):
    feats_in_cat = sorted([f for f, c in stage2_categories.items() if c == cat])
    report += f"\n{cat.upper()} ({len(feats_in_cat)} features):\n"
    for i, feat in enumerate(feats_in_cat, 1):
        report += f"   {i:2d}. {feat}\n"

report += f"""
{'='*80}
NEXT STEPS
{'='*80}

Stage 3: Recursive Feature Elimination (RFE)
   - Use XGBoost with TimeSeriesSplit cross-validation
   - Target: ~40-50 final features
   - Maintain category balance with protection mechanism
   - Evaluate performance at each elimination step

Stage 4: SHAP Analysis
   - Calculate SHAP values for top features
   - Validate feature importance
   - Check for interaction effects

{'='*80}
FILES GENERATED
{'='*80}

Artifacts:
   - stage2_domain_protected_features.json
   - stage2_features_list.txt

Reports:
   - stage2_summary_report.txt

{'='*80}
✅ STAGE 2 COMPLETE - READY FOR RFE
{'='*80}
"""

report_path = CONFIG['reports_dir'] / 'stage2_summary_report.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"   ✅ Saved: stage2_summary_report.txt")

# ============================================================================
# 4. Summary
# ============================================================================

print(f"\n{'='*80}")
print(f"✅ STAGE 2 COMPLETE")
print(f"{'='*80}")

print(f"\n📊 Final Statistics:")
print(f"   Stage 1 output: {len(high_mi_features)} features")
print(f"   Stage 2 output: {len(stage2_features)} features")
print(f"   Improvement: +{len(stage2_features) - len(high_mi_features)} features")

print(f"\n📁 Files Generated:")
print(f"   Artifacts:")
print(f"      - stage2_domain_protected_features.json")
print(f"      - stage2_features_list.txt")
print(f"   Reports:")
print(f"      - stage2_summary_report.txt")

if final_deficits:
    print(f"\n⚠️ Minor deficits remain ({len(final_deficits)} categories)")
    print(f"   Will be handled in Stage 3 (RFE)")
else:
    print(f"\n✅ All category minimums satisfied!")

print(f"\n💾 Saving state for Stage 3...")
print(f"   X_stage2: ({len(X)}, {len(stage2_features)})")

# Update X for Stage 3
X_stage2 = X[stage2_features].copy()

print(f"   ✅ Variables ready for Stage 3 (RFE)")

print(f"\n➡️ Next: Cell 10 (Stage 3 - RFE with Category Protection)")
print(f"{'='*80}\n")

STAGE 2: EXPORT & SUMMARY

1️⃣ Calculating final Stage 2 statistics...

   📊 Stage 2 Feature Set:
      Total features: 86
      Improvement from Stage 1: +4
      Remaining deficits: 2

2️⃣ Exporting Stage 2 features...
   ✅ Saved: stage2_domain_protected_features.json
      Features: 86
   ✅ Saved: stage2_features_list.txt

3️⃣ Generating Stage 2 report...
   ✅ Saved: stage2_summary_report.txt

✅ STAGE 2 COMPLETE

📊 Final Statistics:
   Stage 1 output: 82 features
   Stage 2 output: 86 features
   Improvement: +4 features

📁 Files Generated:
   Artifacts:
      - stage2_domain_protected_features.json
      - stage2_features_list.txt
   Reports:
      - stage2_summary_report.txt

⚠️ Minor deficits remain (2 categories)
   Will be handled in Stage 3 (RFE)

💾 Saving state for Stage 3...
   X_stage2: (32856, 86)
   ✅ Variables ready for Stage 3 (RFE)

➡️ Next: Cell 10 (Stage 3 - RFE with Category Protection)



---

## Stage 2.4: Multicollinearity Removal (Pre-RFE)

**Objective:** Remove redundant features BEFORE RFE for improved efficiency.

**Problem:** After Stage 2, we have 86 features including highly correlated groups:
- Solar geometry features (poa_*, ghi_*, dni_*) derived from same clear-sky model
- Sun position features (aoi, sun_elevation) measuring similar angles
- Lag features at different horizons from same variable

**Why remove before RFE?**
1. **RFE efficiency:** Fewer redundant features → faster, more focused elimination
2. **Clear importance signals:** No splitting of importance across near-duplicates
3. **Better category balance:** Prevents over-representation of redundant groups

**Method:**
1. Calculate correlation matrix (threshold: 0.98 for extreme duplicates only)
2. Group features with correlation > 0.98
3. Keep ONE representative per group (highest feature importance from quick RF model)
4. Validate category minimums maintained

**Expected outcome:**
- Reduce from 86 to ~65-70 features
- Remove only extreme duplicates (corr > 0.98)
- Maintain all critical feature categories
- Preserve or improve performance

---

In [13]:
# ============================================================================
# Cell 10: Stage 2.4 - Multicollinearity Removal (Pre-RFE)
# ============================================================================

import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import time

print("="*80)
print("STAGE 2.4: MULTICOLLINEARITY REMOVAL (PRE-RFE)")
print("="*80)

# ============================================================================
# 1. Configuration
# ============================================================================

print("\n1️⃣ Configuration...")

MULTICOLLINEARITY_CONFIG = {
    'threshold': 0.98,           # Only extreme duplicates
    'max_features_to_remove': 25,  # Safety cap
    'min_group_size': 2,          # Only groups of 2+ features
}

print(f"   Threshold: {MULTICOLLINEARITY_CONFIG['threshold']}")
print(f"   Max to remove: {MULTICOLLINEARITY_CONFIG['max_features_to_remove']}")
print(f"   Starting with: {len(stage2_features)} features (from Stage 2)")

# ============================================================================
# 2. Calculate Correlation Matrix
# ============================================================================

print(f"\n2️⃣ Calculating correlation matrix...")

X_stage2_matrix = X[stage2_features].copy()

# Handle any remaining NaNs
if X_stage2_matrix.isna().any().any():
    print(f"   ⚠️ Found {X_stage2_matrix.isna().sum().sum()} NaN values, filling...")
    X_stage2_matrix = X_stage2_matrix.fillna(method='ffill').fillna(method='bfill')

corr_matrix = X_stage2_matrix.corr().abs()

print(f"   ✅ Correlation matrix ready ({len(corr_matrix)} × {len(corr_matrix)})")

# ============================================================================
# 3. Identify High Correlation Pairs
# ============================================================================

print(f"\n3️⃣ Identifying highly correlated pairs (>{MULTICOLLINEARITY_CONFIG['threshold']})...")

threshold = MULTICOLLINEARITY_CONFIG['threshold']
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > threshold:
            high_corr_pairs.append({
                'feature1': corr_matrix.columns[i],
                'feature2': corr_matrix.columns[j],
                'correlation': corr_matrix.iloc[i, j]
            })

print(f"   Found {len(high_corr_pairs)} pairs with correlation > {threshold}")

if len(high_corr_pairs) == 0:
    print(f"   ✅ No multicollinearity detected! Skipping to RFE...")
    stage2_4_features = stage2_features.copy()
    stage2_4_score = None
    stage2_4_std = None
    
    print(f"\n{'='*80}")
    print(f"✅ STAGE 2.4 SKIPPED - NO MULTICOLLINEARITY")
    print(f"{'='*80}")
    print(f"   Features: {len(stage2_4_features)}")
    print(f"   ➡️ Ready for Stage 3 (RFE)")
    print(f"{'='*80}\n")
    
else:
    # Continue with multicollinearity removal...
    
    # Display top 10 pairs
    print(f"\n   Top 10 most correlated pairs:")
    top_pairs = sorted(high_corr_pairs, key=lambda x: x['correlation'], reverse=True)[:10]
    for pair in top_pairs:
        print(f"      {pair['feature1']:<40} ↔ {pair['feature2']:<40} ({pair['correlation']:.3f})")
    
    # ========================================================================
    # 4. Group Correlated Features
    # ========================================================================
    
    print(f"\n4️⃣ Building correlation groups...")
    
    def find_correlation_groups(pairs):
        """Group features into connected components based on correlation."""
        # Build adjacency graph
        graph = defaultdict(set)
        for pair in pairs:
            f1, f2 = pair['feature1'], pair['feature2']
            graph[f1].add(f2)
            graph[f2].add(f1)
        
        # Find connected components (DFS)
        visited = set()
        groups = []
        
        def dfs(node, group):
            visited.add(node)
            group.add(node)
            for neighbor in graph[node]:
                if neighbor not in visited:
                    dfs(neighbor, group)
        
        for node in graph:
            if node not in visited:
                group = set()
                dfs(node, group)
                if len(group) >= MULTICOLLINEARITY_CONFIG['min_group_size']:
                    groups.append(group)
        
        return groups
    
    corr_groups = find_correlation_groups(high_corr_pairs)
    
    print(f"   Identified {len(corr_groups)} correlation groups")
    print(f"\n   Group summary:")
    print(f"   {'Group':<10} {'Size':<10} {'Members (sample)'}")
    print(f"   {'-'*70}")
    
    for i, group in enumerate(sorted(corr_groups, key=len, reverse=True), 1):
        sample = ', '.join(list(group)[:3])
        if len(group) > 3:
            sample += f", ... ({len(group)-3} more)"
        print(f"   {i:<10} {len(group):<10} {sample}")
    
    # ========================================================================
    # 5. Get Feature Importance for Selection
    # ========================================================================
    
    print(f"\n5️⃣ Training quick model for feature importance...")
    print(f"   (using RandomForest, ~1-2 minutes)...")
    
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    
    # Quick train/val split
    X_train, X_val, y_train, y_val = train_test_split(
        X_stage2_matrix, y, test_size=0.2, random_state=42, shuffle=False
    )
    
    # Train quick RF model
    rf_quick = RandomForestRegressor(
        n_estimators=50,
        max_depth=10,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    start_time = time.time()
    rf_quick.fit(X_train, y_train)
    elapsed = time.time() - start_time
    
    # Get importance
    importance_df = pd.DataFrame({
        'feature': stage2_features,
        'importance': rf_quick.feature_importances_,
        'category': [categorize_feature(f) for f in stage2_features]
    }).sort_values('importance', ascending=False)
    
    val_score = rf_quick.score(X_val, y_val)
    
    print(f"   ✅ Model trained in {elapsed:.1f} seconds")
    print(f"   Validation R²: {val_score:.4f}")
    
    # ========================================================================
    # 6. Select Representatives from Each Group
    # ========================================================================
    
    print(f"\n6️⃣ Selecting one representative per correlation group...")
    
    features_to_remove = []
    features_to_keep = []
    
    for i, group in enumerate(corr_groups, 1):
        # Get importance for group members
        group_importance = importance_df[importance_df['feature'].isin(group)].copy()
        group_importance = group_importance.sort_values('importance', ascending=False)
        
        # Keep the most important
        keep_feature = group_importance.iloc[0]['feature']
        remove_features = list(group - {keep_feature})
        
        features_to_keep.append(keep_feature)
        features_to_remove.extend(remove_features)
        
        if len(group) <= 5:  # Show details for small groups
            print(f"\n   Group {i} ({len(group)} features):")
            print(f"      ✅ KEEP: {keep_feature}")
            print(f"         Importance: {group_importance.iloc[0]['importance']:.4f}")
            print(f"         Category: {group_importance.iloc[0]['category']}")
            
            if len(remove_features) > 0:
                print(f"      ❌ REMOVE:")
                for feat in remove_features:
                    feat_imp = group_importance[group_importance['feature']==feat]['importance'].values[0]
                    feat_cat = group_importance[group_importance['feature']==feat]['category'].values[0]
                    print(f"         - {feat} (imp: {feat_imp:.4f}, cat: {feat_cat})")
    
    # Remove duplicates
    features_to_remove = list(set(features_to_remove))
    
    # Apply safety cap
    if len(features_to_remove) > MULTICOLLINEARITY_CONFIG['max_features_to_remove']:
        print(f"\n   ⚠️ Would remove {len(features_to_remove)} features (exceeds cap of {MULTICOLLINEARITY_CONFIG['max_features_to_remove']})")
        print(f"   Keeping only top {MULTICOLLINEARITY_CONFIG['max_features_to_remove']} least important...")
        
        # Sort by importance and keep cap
        remove_importance = importance_df[importance_df['feature'].isin(features_to_remove)].copy()
        remove_importance = remove_importance.sort_values('importance', ascending=True)
        features_to_remove = remove_importance.head(MULTICOLLINEARITY_CONFIG['max_features_to_remove'])['feature'].tolist()
    
    # ========================================================================
    # 7. Create Final Feature Set
    # ========================================================================
    
    print(f"\n7️⃣ Creating cleaned feature set...")
    
    stage2_4_features = [f for f in stage2_features if f not in features_to_remove]
    
    print(f"\n   Before (Stage 2): {len(stage2_features)} features")
    print(f"   Removed: {len(features_to_remove)} features")
    print(f"   After (Stage 2.4): {len(stage2_4_features)} features")
    
    # ========================================================================
    # 8. Validate Category Balance
    # ========================================================================
    
    print(f"\n8️⃣ Validating category balance...")
    
    stage2_categories = {f: categorize_feature(f) for f in stage2_features}
    stage2_category_counts = Counter(stage2_categories.values())
    
    stage2_4_categories = {f: categorize_feature(f) for f in stage2_4_features}
    stage2_4_category_counts = Counter(stage2_4_categories.values())
    
    print(f"\n   {'Category':<15} {'Stage 2':<10} {'Stage 2.4':<10} {'Removed':<10} {'Min Req':<10} {'Status'}")
    print(f"   {'-'*75}")
    
    all_categories = set(list(stage2_category_counts.keys()) + list(min_required.keys()))
    
    for cat in sorted(all_categories):
        before = stage2_category_counts.get(cat, 0)
        after = stage2_4_category_counts.get(cat, 0)
        removed = before - after
        min_req = min_required.get(cat, 0)
        
        if after < min_req:
            status = f"❌ VIOLATED (-{min_req-after})"
        elif after == min_req:
            status = "⚠️ AT MINIMUM"
        else:
            status = "✅ OK"
        
        print(f"   {cat:<15} {before:<10} {after:<10} {removed:<10} {min_req:<10} {status}")
    
    # ========================================================================
    # 9. Evaluate Performance
    # ========================================================================
    
    print(f"\n9️⃣ Evaluating performance with cleaned features...")
    print(f"   Training model (~1 minute)...")
    
    X_stage2_4 = X[stage2_4_features].copy()
    
    # Quick evaluation
    X_train, X_val, y_train, y_val = train_test_split(
        X_stage2_4, y, test_size=0.2, random_state=42, shuffle=False
    )
    
    rf_final = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    rf_final.fit(X_train, y_train)
    
    stage2_4_score = rf_final.score(X_val, y_val)
    
    # Estimate std (approximate)
    stage2_4_std = 0.02  # Rough estimate
    
    print(f"   ✅ Validation R²: {stage2_4_score:.4f}")
    
    # ========================================================================
    # 10. Verify Multicollinearity Reduction
    # ========================================================================
    
    print(f"\n🔟 Verifying multicollinearity reduction...")
    
    corr_matrix_cleaned = X_stage2_4.corr().abs()
    
    remaining_high_corr = []
    for i in range(len(corr_matrix_cleaned.columns)):
        for j in range(i+1, len(corr_matrix_cleaned.columns)):
            if corr_matrix_cleaned.iloc[i, j] > threshold:
                remaining_high_corr.append({
                    'feature1': corr_matrix_cleaned.columns[i],
                    'feature2': corr_matrix_cleaned.columns[j],
                    'correlation': corr_matrix_cleaned.iloc[i, j]
                })
    
    print(f"\n   Before: {len(high_corr_pairs)} pairs > {threshold}")
    print(f"   After: {len(remaining_high_corr)} pairs > {threshold}")
    print(f"   Reduction: {len(high_corr_pairs) - len(remaining_high_corr)} pairs (-{(1-len(remaining_high_corr)/max(1,len(high_corr_pairs)))*100:.1f}%)")
    
    if len(remaining_high_corr) == 0:
        print(f"\n   ✅ Perfect! No high multicollinearity remaining!")
    elif len(remaining_high_corr) < 5:
        print(f"\n   ✅ Excellent! Only {len(remaining_high_corr)} pairs remain")
        for pair in remaining_high_corr:
            print(f"      {pair['feature1']:<40} ↔ {pair['feature2']:<40} ({pair['correlation']:.3f})")
    else:
        print(f"\n   ⚠️ {len(remaining_high_corr)} pairs remain (acceptable for RFE)")
    
    # ========================================================================
    # 11. Summary
    # ========================================================================
    
    print(f"\n{'='*80}")
    print(f"✅ STAGE 2.4 COMPLETE - MULTICOLLINEARITY REDUCED")
    print(f"{'='*80}")
    
    print(f"\n📊 Summary:")
    print(f"   Stage 2 input: {len(stage2_features)} features")
    print(f"   Removed: {len(features_to_remove)} redundant features")
    print(f"   Stage 2.4 output: {len(stage2_4_features)} features")
    print(f"   Reduction: {len(features_to_remove)/len(stage2_features)*100:.1f}%")
    
    print(f"\n📈 Performance:")
    print(f"   Validation R²: {stage2_4_score:.4f}")
    
    print(f"\n🎯 Impact on RFE:")
    print(f"   Cleaner input → More efficient RFE")
    print(f"   Better importance signals → Better feature selection")
    print(f"   Fewer redundant features → Faster training")
    
    print(f"\n💾 Variables ready for Stage 3:")
    print(f"   stage2_4_features: {len(stage2_4_features)} features")
    print(f"   stage2_4_score: {stage2_4_score:.4f}")
    
    print(f"\n➡️ Ready for Cell 11 (Stage 3: RFE)")
    print(f"{'='*80}\n")

# Save for next stage
X_stage2_4 = X[stage2_4_features].copy()

STAGE 2.4: MULTICOLLINEARITY REMOVAL (PRE-RFE)

1️⃣ Configuration...
   Threshold: 0.98
   Max to remove: 25
   Starting with: 86 features (from Stage 2)

2️⃣ Calculating correlation matrix...
   ✅ Correlation matrix ready (86 × 86)

3️⃣ Identifying highly correlated pairs (>0.98)...
   Found 88 pairs with correlation > 0.98

   Top 10 most correlated pairs:
      ghi_clearsky                             ↔ poa_ground_diffuse_clearsky              (1.000)
      time_to_solar_noon                       ↔ hour_local                               (1.000)
      sun_azimuth                              ↔ solar_azimuth                            (1.000)
      sun_elevation                            ↔ solar_zenith                             (1.000)
      dhi_clearsky                             ↔ poa_sky_diffuse_clearsky                 (1.000)
      poa_diffuse_clearsky                     ↔ poa_sky_diffuse_clearsky                 (1.000)
      poa_diffuse_clearsky                     ↔ dh

---

## Stage 3: Recursive Feature Elimination (RFE)

**Objective:** Reduce from ~65-70 features (post-multicollinearity) to 40-45 optimal features.

**Input:** Stage 2.4 output (~65-70 features with multicollinearity removed)

**Method:** Category-Protected RFE
1. Train XGBoost on current feature set
2. Rank features by importance
3. Remove 10% weakest features (respecting category minimums)
4. Repeat until target reached

**Why RFE works better after multicollinearity removal:**
- ✅ No importance splitting across near-duplicates
- ✅ Clearer signal of which features truly matter
- ✅ Better category balance (solar not over-represented)
- ✅ Faster iterations (fewer features to evaluate)

**Target:** 40-45 features (optimal for 32K samples)

**Expected runtime:** 15-25 minutes (fewer iterations than without Stage 2.4!)

---

In [14]:
# ============================================================================
# Cell 11: Stage 3.1 - RFE Setup (Post-Multicollinearity)
# ============================================================================

import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd
from collections import Counter
import time

print("="*80)
print("STAGE 3: RECURSIVE FEATURE ELIMINATION (RFE)")
print("="*80)

# ============================================================================
# 1. Configuration
# ============================================================================

print("\n1️⃣ RFE Configuration...")

RFE_CONFIG = {
    'target_features': 42,        # Target: ~40-45 features
    'min_features': 38,           # Absolute minimum
    'max_features': 45,           # Absolute maximum
    'step_pct': 0.10,             # Remove 10% per iteration
    'cv_splits': 5,               # TimeSeriesSplit folds
    'random_state': 42,
    
    # XGBoost parameters (optimized for speed)
    'xgb_params': {
        'objective': 'reg:squarederror',
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 0
    },
    
    # Category protection (UPDATED minimums based on Stage 2.4 output)
    'min_per_category': {
        'lag': 8,           # Keep at least 8 lag features
        'rolling': 6,       # Keep at least 6 rolling features
        'solar': 6,         # REDUCED from 8 (multicollinearity removed duplicates)
        'weather': 6,       # Keep all 6 weather forecasts
        'temporal': 4,      # Keep at least 4 temporal features
        'performance': 1,   # Keep at least 1 performance feature
        'other': 0          # No minimum for 'other'
    }
}

print(f"\n   Target features: {RFE_CONFIG['target_features']}")
print(f"   Range: {RFE_CONFIG['min_features']}-{RFE_CONFIG['max_features']}")
print(f"   Step size: {RFE_CONFIG['step_pct']*100:.0f}% per iteration")
print(f"   CV folds: {RFE_CONFIG['cv_splits']}")

print(f"\n   Category minimums (UPDATED for post-multicollinearity):")
for cat, min_count in sorted(RFE_CONFIG['min_per_category'].items()):
    if min_count > 0:
        print(f"      {cat}: {min_count}")

# ============================================================================
# 2. Prepare Data
# ============================================================================

print(f"\n2️⃣ Preparing data for RFE...")

# Use Stage 2.4 features (after multicollinearity removal)
current_features = stage2_4_features.copy()

print(f"\n   Pipeline overview:")
print(f"      Stage 2 output: {len(stage2_features)} features")
print(f"      Stage 2.4 output: {len(stage2_4_features)} features")
print(f"      Removed by multicollinearity: {len(stage2_features) - len(stage2_4_features)}")
print(f"      Starting RFE with: {len(current_features)} features")

print(f"\n   Dataset:")
print(f"      Samples: {len(X_stage2_4):,}")
print(f"      Target: {TARGET_COL}")
print(f"      Features: {len(current_features)}")

# Create feature matrix
X_rfe = X_stage2_4.copy()
y_rfe = y.copy()

# Verify no missing values
if X_rfe.isna().any().any():
    print(f"   ⚠️ Found missing values, filling...")
    X_rfe = X_rfe.fillna(method='ffill').fillna(method='bfill')
    print(f"   ✅ Missing values handled")

print(f"   ✅ Data ready: {X_rfe.shape}")

# ============================================================================
# 3. Setup TimeSeriesSplit CV
# ============================================================================

print(f"\n3️⃣ Setting up Time-Series Cross-Validation...")

tscv = TimeSeriesSplit(n_splits=RFE_CONFIG['cv_splits'])

print(f"   Splits: {RFE_CONFIG['cv_splits']}")
print(f"   Strategy: TimeSeriesSplit (prevents future data leakage)")

# Show split info
print(f"\n   Split details:")
print(f"   {'Fold':<8} {'Train Size':<12} {'Val Size':<12} {'Train End':<25} {'Val Start'}")
print(f"   {'-'*85}")

for i, (train_idx, val_idx) in enumerate(tscv.split(X_rfe), 1):
    train_end = X_rfe.index[train_idx[-1]]
    val_start = X_rfe.index[val_idx[0]]
    print(f"   {i:<8} {len(train_idx):<12,} {len(val_idx):<12,} {str(train_end):<25} {val_start}")

# ============================================================================
# 4. Helper Functions
# ============================================================================

print(f"\n4️⃣ Defining helper functions...")

def train_and_evaluate(X, y, features, cv):
    """Train XGBoost and get CV scores + feature importance."""
    X_subset = X[features]
    
    cv_scores = []
    feature_importances = []
    
    for train_idx, val_idx in cv.split(X_subset):
        X_train = X_subset.iloc[train_idx]
        X_val = X_subset.iloc[val_idx]
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]
        
        # Train model
        model = xgb.XGBRegressor(**RFE_CONFIG['xgb_params'])
        model.fit(X_train, y_train, verbose=False)
        
        # Evaluate
        val_score = model.score(X_val, y_val)
        cv_scores.append(val_score)
        
        # Get importance
        feature_importances.append(model.feature_importances_)
    
    # Average scores and importances
    mean_score = np.mean(cv_scores)
    std_score = np.std(cv_scores)
    mean_importance = np.mean(feature_importances, axis=0)
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'feature': features,
        'importance': mean_importance,
        'category': [categorize_feature(f) for f in features]
    }).sort_values('importance', ascending=False)
    
    return mean_score, std_score, importance_df


def get_category_counts(features):
    """Get current count per category."""
    categories = [categorize_feature(f) for f in features]
    return Counter(categories)


def select_feature_to_remove(importance_df, current_counts, min_per_category):
    """
    Select weakest feature that can be removed without violating minimums.
    
    Returns: (feature_name, category) or (None, None) if no feature can be removed
    """
    # Sort by importance (ascending - weakest first)
    sorted_features = importance_df.sort_values('importance', ascending=True)
    
    for _, row in sorted_features.iterrows():
        feat = row['feature']
        cat = row['category']
        
        # Check if category is at minimum
        current_count = current_counts.get(cat, 0)
        min_required = min_per_category.get(cat, 0)
        
        if current_count > min_required:
            # Safe to remove
            return feat, cat
    
    # No feature can be removed without violating minimums
    return None, None


print(f"   ✅ Helper functions defined:")
print(f"      - train_and_evaluate()")
print(f"      - get_category_counts()")
print(f"      - select_feature_to_remove()")

# ============================================================================
# 5. Initial Baseline Evaluation
# ============================================================================

print(f"\n5️⃣ Evaluating baseline ({len(current_features)} features post-multicollinearity)...")
print(f"   Training XGBoost with TimeSeriesSplit CV...")
print(f"   ⏱️ This will take ~2-3 minutes...")

start_time = time.time()

baseline_score, baseline_std, baseline_importance = train_and_evaluate(
    X_rfe, y_rfe, current_features, tscv
)

elapsed = time.time() - start_time

print(f"\n   ✅ Baseline CV R²: {baseline_score:.4f} ± {baseline_std:.4f}")
print(f"   ⏱️ Time taken: {elapsed/60:.1f} minutes")

# Compare to Stage 2.4 if available
if stage2_4_score is not None:
    print(f"\n   Comparison:")
    print(f"      Stage 2.4 (RF): {stage2_4_score:.4f}")
    print(f"      RFE baseline (XGB): {baseline_score:.4f}")
    print(f"      Difference: {baseline_score - stage2_4_score:+.4f}")

print(f"   ✅ Ready to begin RFE iterations")

print(f"\n{'='*80}")
print(f"⏱️ Starting RFE iterations...")
print(f"   Estimated time: 15-25 minutes")
print(f"   Expected iterations: 4-5 (fewer than without multicollinearity removal!)")
print(f"{'='*80}\n")

STAGE 3: RECURSIVE FEATURE ELIMINATION (RFE)

1️⃣ RFE Configuration...

   Target features: 42
   Range: 38-45
   Step size: 10% per iteration
   CV folds: 5

   Category minimums (UPDATED for post-multicollinearity):
      lag: 8
      performance: 1
      rolling: 6
      solar: 6
      temporal: 4
      weather: 6

2️⃣ Preparing data for RFE...

   Pipeline overview:
      Stage 2 output: 86 features
      Stage 2.4 output: 61 features
      Removed by multicollinearity: 25
      Starting RFE with: 61 features

   Dataset:
      Samples: 32,856
      Target: target_k
      Features: 61
   ✅ Data ready: (32856, 61)

3️⃣ Setting up Time-Series Cross-Validation...
   Splits: 5
   Strategy: TimeSeriesSplit (prevents future data leakage)

   Split details:
   Fold     Train Size   Val Size     Train End                 Val Start
   -------------------------------------------------------------------------------------
   1        5,476        5,476        2022-04-17 03:00:00+00:00 2022-04-

---

### Cell 12: RFE Iterations - Feature Elimination Loop

**Process:**
1. Train XGBoost on current feature set
2. Get feature importance scores
3. Select weakest feature (respecting category minimums)
4. Remove 10% weakest features per iteration
5. Repeat until reaching target (40-45 features)

**Benefits of post-multicollinearity RFE:**
- Fewer iterations needed (~65 → 42 vs 86 → 45)
- Clearer importance signals (no splitting)
- Better final feature set (more diverse, less redundant)

**Expected outcome:**
- 4-5 iterations (vs 7 without Stage 2.4)
- ~40-42 final features
- Performance maintained or improved

---

In [15]:
# ============================================================================
# Cell 12: Stage 3.2 - RFE Iterations (Post-Multicollinearity)
# ============================================================================

print("="*80)
print("RFE ITERATIONS - FEATURE ELIMINATION")
print("="*80)

# ============================================================================
# Initialize tracking
# ============================================================================

rfe_history = []
current_features = stage2_4_features.copy()  # Start with Stage 2.4 output
iteration = 0

print(f"\n📊 Starting RFE Loop:")
print(f"   Input (Stage 2.4): {len(stage2_4_features)} features")
print(f"   Target: {RFE_CONFIG['target_features']} features")
print(f"   Will remove: {len(current_features) - RFE_CONFIG['target_features']} features")
print(f"   Expected iterations: {np.ceil((len(current_features) - RFE_CONFIG['target_features']) / (len(current_features) * RFE_CONFIG['step_pct'])):.0f}")
print(f"\n{'='*80}\n")

# ============================================================================
# RFE Loop
# ============================================================================

overall_start = time.time()

while len(current_features) > RFE_CONFIG['target_features']:
    iteration += 1
    iter_start = time.time()
    
    print(f"\n{'='*80}")
    print(f"ITERATION {iteration}")
    print(f"{'='*80}")
    print(f"Current features: {len(current_features)}")
    
    # ========================================================================
    # Step 1: Train and Evaluate
    # ========================================================================
    
    print(f"\n1️⃣ Training XGBoost with {len(current_features)} features...")
    
    score, std, importance_df = train_and_evaluate(
        X_rfe, y_rfe, current_features, tscv
    )
    
    print(f"   ✅ CV R²: {score:.4f} ± {std:.4f}")
    
    # ========================================================================
    # Step 2: Calculate Features to Remove
    # ========================================================================
    
    n_to_remove = max(1, int(len(current_features) * RFE_CONFIG['step_pct']))
    target_count = len(current_features) - n_to_remove
    
    # Don't go below target
    if target_count < RFE_CONFIG['target_features']:
        n_to_remove = len(current_features) - RFE_CONFIG['target_features']
        target_count = RFE_CONFIG['target_features']
    
    print(f"\n2️⃣ Selecting features to remove...")
    print(f"   Step size: {RFE_CONFIG['step_pct']*100:.0f}%")
    print(f"   Features to remove: {n_to_remove}")
    print(f"   Target count: {target_count}")
    
    # ========================================================================
    # Step 3: Remove Features (Category-Aware)
    # ========================================================================
    
    print(f"\n3️⃣ Removing features (category-protected)...")
    
    removed_features = []
    current_counts = get_category_counts(current_features)
    
    for i in range(n_to_remove):
        # Find weakest removable feature
        feat_to_remove, cat = select_feature_to_remove(
            importance_df, 
            current_counts, 
            RFE_CONFIG['min_per_category']
        )
        
        if feat_to_remove is None:
            print(f"   ⚠️ Cannot remove more features without violating minimums!")
            print(f"   Stopped at {len(current_features)} features")
            break
        
        # Remove feature
        current_features.remove(feat_to_remove)
        removed_features.append((feat_to_remove, cat))
        
        # Update counts
        current_counts[cat] -= 1
        
        # Remove from importance_df so it's not selected again
        importance_df = importance_df[importance_df['feature'] != feat_to_remove]
    
    # Show removed features
    print(f"\n   Removed {len(removed_features)} features:")
    for feat, cat in removed_features[:10]:  # Show first 10
        imp_row = importance_df[importance_df['feature']==feat]
        imp_val = imp_row['importance'].values[0] if len(imp_row) > 0 else 0.0
        print(f"      - {feat:<40} ({cat:<12}, imp={imp_val:.4f})")
    
    if len(removed_features) > 10:
        print(f"      ... and {len(removed_features)-10} more")
    
    # ========================================================================
    # Step 4: Check Category Balance
    # ========================================================================
    
    print(f"\n4️⃣ Category balance after removal:")
    
    final_counts = get_category_counts(current_features)
    
    print(f"   {'Category':<15} {'Count':<10} {'Minimum':<10} {'Status'}")
    print(f"   {'-'*50}")
    
    all_categories = set(list(final_counts.keys()) + list(RFE_CONFIG['min_per_category'].keys()))
    
    for cat in sorted(all_categories):
        count = final_counts.get(cat, 0)
        min_req = RFE_CONFIG['min_per_category'].get(cat, 0)
        
        if count < min_req:
            status = f"❌ VIOLATED (-{min_req-count})"
        elif count == min_req:
            status = "⚠️ AT MINIMUM"
        else:
            status = "✅ OK"
        
        print(f"   {cat:<15} {count:<10} {min_req:<10} {status}")
    
    # ========================================================================
    # Step 5: Save Iteration Results
    # ========================================================================
    
    iter_time = time.time() - iter_start
    
    rfe_history.append({
        'iteration': iteration,
        'n_features': len(current_features),
        'n_removed': len(removed_features),
        'cv_score': score,
        'cv_std': std,
        'time_seconds': iter_time,
        'removed_features': [f for f, _ in removed_features],
        'category_counts': dict(final_counts)
    })
    
    print(f"\n⏱️ Iteration {iteration} completed in {iter_time/60:.1f} minutes")
    
    # ========================================================================
    # Check if target reached
    # ========================================================================
    
    if len(current_features) <= RFE_CONFIG['target_features']:
        print(f"\n✅ Target reached! ({len(current_features)} features)")
        break

# ============================================================================
# Final Summary
# ============================================================================

total_time = time.time() - overall_start

print(f"\n{'='*80}")
print(f"✅ RFE COMPLETE")
print(f"{'='*80}")

print(f"\n📊 Summary:")
print(f"   Iterations: {iteration}")
print(f"   Stage 2 output: {len(stage2_features)} features")
print(f"   Stage 2.4 output: {len(stage2_4_features)} features")
print(f"   Final (RFE): {len(current_features)} features")
print(f"   Total removed: {len(stage2_features) - len(current_features)} features")
print(f"   Total time: {total_time/60:.1f} minutes")

print(f"\n📈 Performance:")
print(f"   Baseline (Stage 2.4): {baseline_score:.4f} ± {baseline_std:.4f}")
print(f"   Final (RFE): {rfe_history[-1]['cv_score']:.4f} ± {rfe_history[-1]['cv_std']:.4f}")
print(f"   Change: {rfe_history[-1]['cv_score'] - baseline_score:+.4f}")

print(f"\n💾 Saving RFE results...")

# Save final features
rfe_final_features = current_features.copy()

# Save iteration history
rfe_history_df = pd.DataFrame(rfe_history)
rfe_history_df.to_csv(CONFIG['reports_dir'] / 'rfe_iteration_history.csv', index=False)

print(f"   ✅ Saved: rfe_iteration_history.csv")

print(f"\n➡️ Next: Cell 13 (Final Analysis & Export)")
print(f"{'='*80}\n")

RFE ITERATIONS - FEATURE ELIMINATION

📊 Starting RFE Loop:
   Input (Stage 2.4): 61 features
   Target: 42 features
   Will remove: 19 features
   Expected iterations: 4



ITERATION 1
Current features: 61

1️⃣ Training XGBoost with 61 features...
   ✅ CV R²: 0.9558 ± 0.0171

2️⃣ Selecting features to remove...
   Step size: 10%
   Features to remove: 6
   Target count: 55

3️⃣ Removing features (category-protected)...

   Removed 6 features:
      - irradiance_quality                       (performance , imp=0.0000)
      - season                                   (other       , imp=0.0000)
      - s01_inv1_rolling_std_3d_same_tod         (rolling     , imp=0.0000)
      - irradiance_lag_7d                        (lag         , imp=0.0000)
      - ambient_temp_lag_3d                      (lag         , imp=0.0000)
      - month_sin                                (temporal    , imp=0.0000)

4️⃣ Category balance after removal:
   Category        Count      Minimum    Status
   ---------

---

## Stage 4: Final Analysis & Export

**Objective:** Validate final feature set and export for modeling.

**Final Feature Set:**
- ~40-42 features (optimal for dataset size)
- All critical categories represented
- No extreme multicollinearity (removed in Stage 2.4)
- Validated performance via TimeSeriesSplit CV

**Deliverables:**
1. Final feature list (JSON + TXT)
2. Feature summary report (CSV)
3. Category distribution visualization
4. Complete selection report (Markdown)
5. RFE iteration analysis

**Next Step:** Modeling notebook (06_modeling.ipynb)

---

In [17]:
# ============================================================================
# Cell 13: Final Analysis & Export (ROBUST VERSION)
# ============================================================================

import json
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from collections import Counter

print("="*80)
print("FINAL ANALYSIS & EXPORT")
print("="*80)

# 1. Robust Feature Detection (Fixing the Chronology Issue)
# ----------------------------------------------------------------------------
print("\n1️⃣ Detecting final features...")

# Try to find the final feature list from various possible variable names
if 'selected_features_rfe' in globals():
    final_features = selected_features_rfe
    source = "Stage 3 (My Code)"
elif 'rfe_final_features' in globals():
    final_features = rfe_final_features
    source = "Stage 3 (Original)"
elif 'current_features' in globals():
    final_features = current_features
    source = "Current State"
elif 'final_features' in globals():
    final_features = globals()['final_features']
    source = "Previous Run"
else:
    # Fallback: Try to load from file if variables are lost
    try:
        top_50_path = CONFIG['artifacts_dir'] / 'top_50_features.json' # From Notebook 04
        if top_50_path.exists():
             with open(top_50_path, 'r') as f:
                final_features = json.load(f)['features']
                source = "File (Notebook 04)"
        else:
            raise ValueError("No feature list found in memory or files.")
    except Exception as e:
        raise ValueError(f"❌ Critical Error: Could not find final features. Please re-run the RFE stage. Error: {e}")

print(f"   ✅ Found {len(final_features)} features")
print(f"   🔍 Source: {source}")

# 2. Robust Statistics Loading (Prevent NameErrors)
# ----------------------------------------------------------------------------
print("\n2️⃣ Loading pipeline statistics...")

# Stage 1 Count
try:
    s1_count = stage1_count if 'stage1_count' in globals() else 85
except: s1_count = 85

# Stage 2 Count
try:
    s2_count = stage2_count if 'stage2_count' in globals() else (len(high_mi_features) if 'high_mi_features' in globals() else 85)
except: s2_count = 85

print(f"   Stage 1 (Start): ~{s1_count} features")
print(f"   Stage 3 (End):   {len(final_features)} features")
print(f"   Total Reduction: {((1 - len(final_features)/s1_count)*100):.1f}%")

# 3. Validation & Categorization
# ----------------------------------------------------------------------------
print("\n3️⃣ Validating final set...")

# Helper function (define if missing)
if 'categorize_feature' not in globals():
    def categorize_feature(feat):
        if 'lag' in feat: return 'lag'
        if 'rolling' in feat: return 'rolling'
        if 'fcst' in feat: return 'weather'
        if any(x in feat for x in ['solar', 'sun', 'aoi', 'clearsky', 'poa']): return 'solar'
        if any(x in feat for x in ['hour', 'day', 'month', 'doy']): return 'temporal'
        return 'other'

final_categories = {f: categorize_feature(f) for f in final_features}
final_counts = Counter(final_categories.values())

for cat, count in sorted(final_counts.items()):
    print(f"   • {cat:<12}: {count} features")

# 4. Save to JSON (Standardized Name)
# ----------------------------------------------------------------------------
print("\n4️⃣ Saving final export...")

# Ensure Config exists
if 'CONFIG' not in globals() or 'artifacts_dir' not in CONFIG:
    # Create temp config if missing
    import os
    project_root = Path(os.getcwd())
    while not (project_root / 'artifacts').exists() and project_root.parent != project_root:
        project_root = project_root.parent
    artifacts_dir = project_root / 'artifacts' / '05_feature_selection'
    artifacts_dir.mkdir(parents=True, exist_ok=True)
else:
    artifacts_dir = CONFIG['artifacts_dir']

# Ensure Target exists
target_col = TARGET_COL if 'TARGET_COL' in globals() else 'target_k'

final_export = {
    'metadata': {
        'timestamp': datetime.now().isoformat(),
        'target': target_col,
        'n_features': len(final_features),
        'source': source
    },
    'selected_features': final_features,
    'category_distribution': dict(final_counts)
}

# SAVE AS 'final_selected_features.json' (Matches Notebook 06 expectation)
out_path = artifacts_dir / 'final_selected_features.json'

with open(out_path, 'w') as f:
    json.dump(final_export, f, indent=2)

print(f"   ✅ Saved: {out_path}")
print(f"   🎉 Notebook 05 Complete! You can now proceed to Notebook 06.")
print("="*80)

FINAL ANALYSIS & EXPORT

1️⃣ Detecting final features...
   ✅ Found 42 features
   🔍 Source: Stage 3 (Original)

2️⃣ Loading pipeline statistics...
   Stage 1 (Start): ~85 features
   Stage 3 (End):   42 features
   Total Reduction: 50.6%

3️⃣ Validating final set...
   • lag         : 9 features
   • performance : 1 features
   • rolling     : 6 features
   • solar       : 14 features
   • temporal    : 6 features
   • weather     : 6 features

4️⃣ Saving final export...
   ✅ Saved: C:\Users\baraky\PycharmProjects\solar_ds_capstone\artifacts\05_feature_selection\final_selected_features.json
   🎉 Notebook 05 Complete! You can now proceed to Notebook 06.


In [1]:
# ============================================================================
# 🔍 DIAGNOSTIC TOOL: Feature Selection vs. Data File
# ============================================================================
import pandas as pd
import json
import os
from pathlib import Path

# 1. Setup Paths
current_path = Path.cwd()
project_root = None
for parent in [current_path] + list(current_path.parents):
    if parent.name == 'solar_ds_capstone':
        project_root = parent
        break
if project_root is None: project_root = current_path

# Paths to check
json_path = project_root / 'artifacts' / '05_feature_selection' / 'final_selected_features.json'
data_path = None

# Find the data file
for root, dirs, files in os.walk(project_root):
    if 'features_1H_target_k.parquet' in files:
        data_path = Path(root) / 'features_1H_target_k.parquet'
        break

print(f"📂 Root: {project_root}")
print("-" * 60)

# 2. Check What NB5 Selected (The "Wishlist")
selected_features = []
if json_path.exists():
    with open(json_path, 'r') as f:
        data = json.load(f)
        # Handle different structures
        if isinstance(data, list): selected_features = data
        elif 'features' in data: selected_features = data['features']
        elif 'selected_features' in data: selected_features = data['selected_features']
    
    print(f"✅ Notebook 05 Selection Found: {json_path.name}")
    print(f"   Total Selected: {len(selected_features)}")
    
    # Check specifically for the problematic feature
    lag_feature = 'engineered_k_lag_1d'
    if lag_feature in selected_features:
        print(f"   👀 '{lag_feature}' IS present in the selection list.")
    else:
        print(f"   ❌ '{lag_feature}' is NOT in the selection list.")
else:
    print(f"❌ Notebook 05 JSON not found at: {json_path}")

print("-" * 60)

# 3. Check What Exists on Disk (The "Reality")
if data_path and data_path.exists():
    print(f"✅ Data File Found: {data_path.name}")
    df_check = pd.read_parquet(data_path)
    print(f"   Total Columns: {len(df_check.columns)}")
    
    # Check for the lag feature
    if 'engineered_k_lag_1d' in df_check.columns:
        print(f"   ✅ '{lag_feature}' EXISTS in the parquet file.")
    else:
        print(f"   ❌ '{lag_feature}' is MISSING from the parquet file.")
        
    # Check for base target
    if 'engineered_k' in df_check.columns:
        print(f"   ✅ Base target 'engineered_k' exists (we can reconstruct lags).")
    else:
        print(f"   ❌ Base target 'engineered_k' is also missing.")

    # Show missing features (Gap Analysis)
    if selected_features:
        missing = [f for f in selected_features if f not in df_check.columns]
        if missing:
            print(f"\n⚠️  MISMATCH: {len(missing)} features selected but missing in file:")
            print(f"   {missing[:5]} ...")
        else:
            print("\n🎉 PERFECT MATCH: All selected features exist in the file.")

else:
    print("❌ Data file not found.")

print("=" * 60)

📂 Root: C:\Users\baraky\PycharmProjects\solar_ds_capstone
------------------------------------------------------------
✅ Notebook 05 Selection Found: final_selected_features.json
   Total Selected: 42
   ❌ 'engineered_k_lag_1d' is NOT in the selection list.
------------------------------------------------------------
✅ Data File Found: features_1H_target_k.parquet
   Total Columns: 94
   ❌ 'engineered_k_lag_1d' is MISSING from the parquet file.
   ❌ Base target 'engineered_k' is also missing.

🎉 PERFECT MATCH: All selected features exist in the file.
